### Basic neural activity analysis with single camera tracking
#### analyze the firing rate PC1,2,3
#### making the demo videos
#### analyze the spike triggered pull and gaze ditribution
#### the following detailed analysis focused on pull related behavioral events

In [72]:
import pandas as pd
import numpy as np
from numpy import genfromtxt
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import seaborn
import scipy
import scipy.stats as st
import scipy.io
from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from dPCA import dPCA
import string
import warnings
import pickle
import json

from scipy.ndimage import gaussian_filter1d

import os
import glob
import random
from time import time

from pgmpy.models import BayesianModel
from pgmpy.models import DynamicBayesianNetwork as DBN
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch,BicScore
from pgmpy.base import DAG
import networkx as nx

from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

from scipy.ndimage import label


### function - get body part location for each pair of cameras

In [73]:
from ana_functions.body_part_locs_eachpair import body_part_locs_eachpair
from ana_functions.body_part_locs_singlecam import body_part_locs_singlecam

### function - align the two cameras

In [74]:
from ana_functions.camera_align import camera_align       

### function - merge the two pairs of cameras

In [75]:
from ana_functions.camera_merge import camera_merge

### function - find social gaze time point

In [76]:
from ana_functions.find_socialgaze_timepoint import find_socialgaze_timepoint
from ana_functions.find_socialgaze_timepoint_singlecam import find_socialgaze_timepoint_singlecam
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody import find_socialgaze_timepoint_singlecam_wholebody
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody_2 import find_socialgaze_timepoint_singlecam_wholebody_2


### function - define time point of behavioral events

In [77]:
from ana_functions.bhv_events_timepoint import bhv_events_timepoint
from ana_functions.bhv_events_timepoint_singlecam import bhv_events_timepoint_singlecam

### function - plot behavioral events

In [78]:
from ana_functions.plot_bhv_events import plot_bhv_events
from ana_functions.plot_bhv_events_levertube import plot_bhv_events_levertube
from ana_functions.plot_continuous_bhv_var_singlecam import plot_continuous_bhv_var_singlecam
from ana_functions.draw_self_loop import draw_self_loop
import matplotlib.patches as mpatches 
from matplotlib.collections import PatchCollection

### function - plot inter-pull interval

In [79]:
from ana_functions.plot_interpull_interval import plot_interpull_interval

### function - make demo videos with skeleton and inportant vectors

In [80]:
from ana_functions.tracking_video_singlecam_demo import tracking_video_singlecam_demo
from ana_functions.tracking_video_singlecam_wholebody_demo import tracking_video_singlecam_wholebody_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_demo import tracking_video_singlecam_wholebody_withNeuron_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo
from ana_functions.tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo

### function - interval between all behavioral events

In [81]:
from ana_functions.bhv_events_interval import bhv_events_interval

### function - spike analysis

In [82]:
from ana_functions.spike_analysis_FR_calculation import spike_analysis_FR_calculation
from ana_functions.plot_spike_triggered_singlecam_bhvevent import plot_spike_triggered_singlecam_bhvevent
from ana_functions.plot_bhv_events_aligned_FR import plot_bhv_events_aligned_FR
from ana_functions.plot_strategy_aligned_FR import plot_strategy_aligned_FR

### function - PCA projection

In [83]:
from ana_functions.PCA_around_bhv_events import PCA_around_bhv_events
from ana_functions.PCA_around_bhv_events_video import PCA_around_bhv_events_video
from ana_functions.confidence_ellipse import confidence_ellipse

### function - train the dynamic bayesian network - multi time lag (3 lags)

In [84]:
from ana_functions.train_DBN_multiLag_withNeuron import train_DBN_multiLag
from ana_functions.train_DBN_multiLag_withNeuron import train_DBN_multiLag_create_df_only
from ana_functions.train_DBN_multiLag_withNeuron import train_DBN_multiLag_training_only
from ana_functions.train_DBN_multiLag_withNeuron import graph_to_matrix
from ana_functions.train_DBN_multiLag_withNeuron import get_weighted_dags
from ana_functions.train_DBN_multiLag_withNeuron import get_significant_edges
from ana_functions.train_DBN_multiLag_withNeuron import threshold_edges
from ana_functions.train_DBN_multiLag_withNeuron import Modulation_Index
from ana_functions.EfficientTimeShuffling import EfficientShuffle
from ana_functions.AicScore import AicScore

### function - other useful functions

In [85]:
# for defining the meaningful social gaze (the continuous gaze distribution that is closest to the pull) 
from ana_functions.keep_closest_cluster_single_trial import keep_closest_cluster_single_trial

In [86]:
# get useful information about pulls
from ana_functions.get_pull_infos import get_pull_infos

In [87]:
# use the gaze vector speed and face mass speed to find the pull action start time within IPI
from ana_functions.find_sharp_increases_withinIPI import find_sharp_increases_withinIPI
from ana_functions.find_sharp_increases_withinIPI import find_sharp_increases_withinIPI_dual_speed

## Analyze each session

### prepare the basic behavioral data (especially the time stamps for each bhv events)

In [88]:
# instead of using gaze angle threshold, use the target rectagon to deside gaze info
# ...need to update
sqr_thres_tubelever = 75 # draw the square around tube and lever
sqr_thres_face = 1.15 # a ratio for defining face boundary
sqr_thres_body = 4 # how many times to enlongate the face box boundry to the body


# get the fps of the analyzed video
fps = 30

# get the fs for neural recording
fs_spikes = 20000
fs_lfp = 1000

# frame number of the demo video
nframes = 0.5*30 # second*30fps
# nframes = 45*30 # second*30fps

# re-analyze the video or not
reanalyze_video = 0
redo_anystep = 0

# if use onset of the first increase after min
doOnsetAfterMin = 0
if not doOnsetAfterMin:
    doOnsetAfterMin_suffix = ''
elif doOnsetAfterMin:
    doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'

# do OFC sessions or DLPFC sessions
do_OFC = 0
do_DLPFC  = 1
if do_OFC:
    savefile_sufix = '_OFCs'
elif do_DLPFC:
    savefile_sufix = '_DLPFCs'
else:
    savefile_sufix = ''
    
# all the videos (no misaligned ones)
# aligned with the audio
# get the session start time from "videosound_bhv_sync.py/.ipynb"
# currently the session_start_time will be manually typed in. It can be updated after a better method is used


# dodson ginger
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                                    '20240531_Dodson_MC',
                                    '20240603_Dodson_MC_and_SR',
                                    '20240603_Dodson_MC_and_SR',
                                    '20240604_Dodson_MC',
                                    '20240605_Dodson_MC_and_SR',
                                    '20240605_Dodson_MC_and_SR',
                                    '20240606_Dodson_MC_and_SR',
                                    '20240606_Dodson_MC_and_SR',
                                    '20240607_Dodson_SR',
                                    '20240610_Dodson_MC',
                                    '20240611_Dodson_SR',
                                    '20240612_Dodson_MC',
                                    '20240613_Dodson_SR',
                                    '20240620_Dodson_SR',
                                    '20240719_Dodson_MC',
                                        
                                    '20250129_Dodson_MC',
                                    '20250130_Dodson_SR',
                                    '20250131_Dodson_MC',
                                
            
                                    '20250210_Dodson_SR_withKoala',
                                    '20250211_Dodson_MC_withKoala',
                                    '20250212_Dodson_SR_withKoala',
                                    '20250214_Dodson_MC_withKoala',
                                    '20250217_Dodson_SR_withKoala',
                                    '20250218_Dodson_MC_withKoala',
                                    '20250219_Dodson_SR_withKoala',
                                    '20250220_Dodson_MC_withKoala',
                                    '20250224_Dodson_KoalaAL_withKoala',
                                    '20250226_Dodson_MC_withKoala',
                                    '20250227_Dodson_KoalaAL_withKoala',
                                    '20250228_Dodson_DodsonAL_withKoala',
                                    '20250304_Dodson_DodsonAL_withKoala',
                                    '20250305_Dodson_MC_withKoala',
                                    '20250306_Dodson_KoalaAL_withKoala',
                                    '20250307_Dodson_DodsonAL_withKoala',
                                    '20250310_Dodson_MC_withKoala',
                                    '20250312_Dodson_NV_withKoala',
                                    '20250313_Dodson_NV_withKoala',
                                    '20250314_Dodson_NV_withKoala',
            
                                    '20250401_Dodson_MC_withKanga',
                                    '20250402_Dodson_MC_withKanga',
                                    '20250403_Dodson_MC_withKanga',
                                    '20250404_Dodson_SR_withKanga',
                                    '20250407_Dodson_SR_withKanga',
                                    '20250408_Dodson_SR_withKanga',
                                    '20250409_Dodson_MC_withKanga',
            
                                    '20250415_Dodson_MC_withKanga',
                                    # '20250416_Dodson_SR_withKanga', # has to remove from the later analysis, recording has problems
                                    '20250417_Dodson_MC_withKanga',
                                    '20250418_Dodson_SR_withKanga',
                                    '20250421_Dodson_SR_withKanga',
                                    '20250422_Dodson_MC_withKanga',
                                    '20250422_Dodson_SR_withKanga',
            
                                    '20250423_Dodson_MC_withKanga',
                                    '20250423_Dodson_SR_withKanga', 
                                    '20250424_Dodson_NV_withKanga',
                                    '20250424_Dodson_MC_withKanga',
                                    '20250424_Dodson_SR_withKanga',            
                                    '20250425_Dodson_NV_withKanga',
                                    '20250425_Dodson_SR_withKanga',
                                    '20250428_Dodson_NV_withKanga',
                                    '20250428_Dodson_MC_withKanga',
                                    '20250428_Dodson_SR_withKanga',  
                                    '20250429_Dodson_NV_withKanga',
                                    '20250429_Dodson_MC_withKanga',
                                    '20250429_Dodson_SR_withKanga',  
                                    '20250430_Dodson_NV_withKanga',
                                    '20250430_Dodson_MC_withKanga',
                                    '20250430_Dodson_SR_withKanga',  
            
                                   ]
        task_conditions = [
                            'MC',           
                            'MC',
                            'SR',
                            'MC',
                            'MC',
                            'SR',
                            'MC',
                            'SR',
                            'SR',
                            'MC',
                            'SR',
                            'MC',
                            'SR',
                            'SR',
                            'MC',
                            
                            'MC_withGingerNew',
                            'SR_withGingerNew',
                            'MC_withGingerNew',
            
                            'SR_withKoala',
                            'MC_withKoala',
                            'SR_withKoala',
                            'MC_withKoala',
                            'SR_withKoala',
                            'MC_withKoala',
                            'SR_withKoala',
                            'MC_withKoala',
                            'MC_KoalaAuto_withKoala',
                            'MC_withKoala',
                            'MC_KoalaAuto_withKoala',
                            'MC_DodsonAuto_withKoala',
                            'MC_DodsonAuto_withKoala',
                            'MC_withKoala',
                            'MC_KoalaAuto_withKoala',
                            'MC_DodsonAuto_withKoala',
                            'MC_withKoala',
                            'NV_withKoala',
                            'NV_withKoala',
                            'NV_withKoala',

                            'MC_withKanga',
                            'MC_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',
                            'SR_withKanga',
                            'SR_withKanga',
                            'MC_withKanga',
            
                            'MC_withKanga',
                            # 'SR_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',
                            'SR_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',
            
                            'MC_withKanga',
                            'SR_withKanga', 
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',            
                            'NV_withKanga',
                            'SR_withKanga',
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',  
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',  
                            'NV_withKanga',
                            'MC_withKanga',
                            'SR_withKanga',  
                          ]
        dates_list = [
                        '20240531',
                        '20240603_MC',
                        '20240603_SR',
                        '20240604',
                        '20240605_MC',
                        '20240605_SR',
                        '20240606_MC',
                        '20240606_SR',
                        '20240607',
                        '20240610_MC',
                        '20240611',
                        '20240612',
                        '20240613',
                        '20240620',
                        '20240719',
            
                        '20250129',
                        '20250130',
                        '20250131',
            
                        '20250210',
                        '20250211',
                        '20250212',
                        '20250214',
                        '20250217',
                        '20250218',
                        '20250219',
                        '20250220',
                        '20250224',
                        '20250226',
                        '20250227',
                        '20250228',
                        '20250304',
                        '20250305',
                        '20250306',
                        '20250307',
                        '20250310',
                        '20250312',
                        '20250313',
                        '20250314',
            
                        '20250401',
                        '20250402',
                        '20250403',
                        '20250404',
                        '20250407',
                        '20250408',
                        '20250409',
            
                        '20250415',
                        # '20250416',
                        '20250417',
                        '20250418',
                        '20250421',
                        '20250422',
                        '20250422_SR',
            
                        '20250423',
                        '20250423_SR', 
                        '20250424',
                        '20250424_MC',
                        '20250424_SR',            
                        '20250425',
                        '20250425_SR',
                        '20250428_NV',
                        '20250428_MC',
                        '20250428_SR',  
                        '20250429_NV',
                        '20250429_MC',
                        '20250429_SR',  
                        '20250430_NV',
                        '20250430_MC',
                        '20250430_SR',  
                     ]
        videodates_list = [
                            '20240531',
                            '20240603',
                            '20240603',
                            '20240604',
                            '20240605',
                            '20240605',
                            '20240606',
                            '20240606',
                            '20240607',
                            '20240610_MC',
                            '20240611',
                            '20240612',
                            '20240613',
                            '20240620',
                            '20240719',
            
                            '20250129',
                            '20250130',
                            '20250131',
                            
                            '20250210',
                            '20250211',
                            '20250212',
                            '20250214',
                            '20250217',
                            '20250218',          
                            '20250219',
                            '20250220',
                            '20250224',
                            '20250226',
                            '20250227',
                            '20250228',
                            '20250304',
                            '20250305',
                            '20250306',
                            '20250307',
                            '20250310',
                            '20250312',
                            '20250313',
                            '20250314',
            
                            '20250401',
                            '20250402',
                            '20250403',
                            '20250404',
                            '20250407',
                            '20250408',
                            '20250409',
            
                            '20250415',
                            # '20250416',
                            '20250417',
                            '20250418',
                            '20250421',
                            '20250422',
                            '20250422_SR',
            
                            '20250423',
                            '20250423_SR', 
                            '20250424',
                            '20250424_MC',
                            '20250424_SR',            
                            '20250425',
                            '20250425_SR',
                            '20250428_NV',
                            '20250428_MC',
                            '20250428_SR',  
                            '20250429_NV',
                            '20250429_MC',
                            '20250429_SR',  
                            '20250430_NV',
                            '20250430_MC',
                            '20250430_SR',  
            
                          ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [ 
                                0.00,
                                340,
                                340,
                                72.0,
                                60.1,
                                60.1,
                                82.2,
                                82.2,
                                35.8,
                                0.00,
                                29.2,
                                35.8,
                                62.5,
                                71.5,
                                54.4,
            
                                0.00,
                                0.00,
                                0.00,
            
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
            
                                0.00,
                                0.00,
                                73.5,
                                0.00,
                                76.1,
                                81.5,
                                0.00,
            
                                363,
                                # 0.00,
                                79.0,
                                162.6,
                                231.9,
                                109,
                                0.00,
            
                                0.00,
                                0.00, 
                                0.00,
                                0.00,
                                0.00,          
                                0.00,
                                93.0,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00, 
                                0.00,
                                274.4,
                                0.00,
            
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0',# 'Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson',# 'dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger',
                               'ginger','ginger','ginger','ginger','ginger','ginger','gingerNew','gingerNew','gingerNew',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', # 'kanga', 
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             'Dodson',# 'Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson',
                            ]
        animal2_filenames = ["Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga",
                             'Kanga', # 'Kanga', 
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                            ]
        
    elif do_OFC:
        # pick only five sessions for each conditions
        neural_record_conditions = [
                                     '20231101_Dodson_withGinger_MC',
                                     '20231107_Dodson_withGinger_MC',
                                     '20231122_Dodson_withGinger_MC',
                                     '20231129_Dodson_withGinger_MC',
                                     # '20231101_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                     # '20231107_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                     # '20231122_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                     # '20231129_Dodson_withGinger_SR', # ginger didnot pull or only pulled once
                                   ]
        task_conditions = [
                            'MC',
                            'MC',
                            'MC',
                            'MC',
                            # 'SR',
                            # 'SR',
                            # 'SR',
                            # 'SR',
                          ]
        dates_list = [
                      "20231101_MC",
                      "20231107_MC",
                      "20231122_MC",
                      "20231129_MC",
                      # "20231101_SR",
                      # "20231107_SR",
                      # "20231122_SR",
                      # "20231129_SR",      
                     ]
        videodates_list = dates_list
        session_start_times = [ 
                                 0.00,   
                                 0.00,  
                                 0.00,  
                                 0.00, 
                                 # 0.00,   
                                 # 0.00,  
                                 # 0.00,  
                                 # 0.00, 
                              ] # in second
        kilosortvers = [ 
                         2, 
                         2, 
                         4, 
                         4,
                         # 2, 
                         # 2, 
                         # 4, 
                         # 4,
                       ]
    
        trig_channelnames = ['Dev1/ai0']*np.shape(dates_list)[0]
        animal1_fixedorders = ['dodson']*np.shape(dates_list)[0]
        recordedanimals = animal1_fixedorders
        animal2_fixedorders = ['ginger']*np.shape(dates_list)[0]

        animal1_filenames = ["Dodson"]*np.shape(dates_list)[0]
        animal2_filenames = ["Ginger"]*np.shape(dates_list)[0]

    
# dannon kanga
if 0:
    if do_DLPFC:
        neural_record_conditions = [
                                     '20240508_Kanga_SR',
                                     '20240509_Kanga_MC',
                                     '20240513_Kanga_MC',
                                     '20240514_Kanga_SR',
                                     '20240523_Kanga_MC',
                                     '20240524_Kanga_SR',
                                     '20240606_Kanga_MC',
                                     '20240613_Kanga_MC_DannonAuto',
                                     '20240614_Kanga_MC_DannonAuto',
                                     '20240617_Kanga_MC_DannonAuto',
                                     '20240618_Kanga_MC_KangaAuto',
                                     '20240619_Kanga_MC_KangaAuto',
                                     '20240620_Kanga_MC_KangaAuto',
                                     '20240621_1_Kanga_NoVis',
                                     '20240624_Kanga_NoVis',
                                     '20240626_Kanga_NoVis',
            
                                     '20240808_Kanga_MC_withGinger',
                                     '20240809_Kanga_MC_withGinger',
                                     '20240812_Kanga_MC_withGinger',
                                     '20240813_Kanga_MC_withKoala',
                                     '20240814_Kanga_MC_withKoala',
                                     '20240815_Kanga_MC_withKoala',
                                     '20240819_Kanga_MC_withVermelho',
                                     '20240821_Kanga_MC_withVermelho',
                                     '20240822_Kanga_MC_withVermelho',
            
                                     '20250415_Kanga_MC_withDodson',
                                     '20250416_Kanga_SR_withDodson',
                                     '20250417_Kanga_MC_withDodson',
                                     '20250418_Kanga_SR_withDodson',
                                     '20250421_Kanga_SR_withDodson',
                                     '20250422_Kanga_MC_withDodson',
                                     '20250422_Kanga_SR_withDodson',
            
                                    '20250423_Kanga_MC_withDodson',
                                    '20250423_Kanga_SR_withDodson', 
                                    '20250424_Kanga_NV_withDodson',
                                    '20250424_Kanga_MC_withDodson',
                                    '20250424_Kanga_SR_withDodson',            
                                    '20250425_Kanga_NV_withDodson',
                                    '20250425_Kanga_SR_withDodson',
                                    '20250428_Kanga_NV_withDodson',
                                    '20250428_Kanga_MC_withDodson',
                                    '20250428_Kanga_SR_withDodson',  
                                    '20250429_Kanga_NV_withDodson',
                                    '20250429_Kanga_MC_withDodson',
                                    '20250429_Kanga_SR_withDodson',  
                                    '20250430_Kanga_NV_withDodson',
                                    '20250430_Kanga_MC_withDodson',
                                    '20250430_Kanga_SR_withDodson',  
                                   ]
        dates_list = [
                      "20240508",
                      "20240509",
                      "20240513",
                      "20240514",
                      "20240523",
                      "20240524",
                      "20240606",
                      "20240613",
                      "20240614",
                      "20240617",
                      "20240618",
                      "20240619",
                      "20240620",
                      "20240621_1",
                      "20240624",
                      "20240626",
            
                      "20240808",
                      "20240809",
                      "20240812",
                      "20240813",
                      "20240814",
                      "20240815",
                      "20240819",
                      "20240821",
                      "20240822",
            
                      "20250415",
                      "20250416",
                      "20250417",
                      "20250418",
                      "20250421",
                      "20250422",
                      "20250422_SR",
            
                        '20250423',
                        '20250423_SR', 
                        '20250424',
                        '20250424_MC',
                        '20250424_SR',            
                        '20250425',
                        '20250425_SR',
                        '20250428_NV',
                        '20250428_MC',
                        '20250428_SR',  
                        '20250429_NV',
                        '20250429_MC',
                        '20250429_SR',  
                        '20250430_NV',
                        '20250430_MC',
                        '20250430_SR',  
                     ]
        videodates_list = dates_list
        task_conditions = [
                             'SR',
                             'MC',
                             'MC',
                             'SR',
                             'MC',
                             'SR',
                             'MC',
                             'MC_DannonAuto',
                             'MC_DannonAuto',
                             'MC_DannonAuto',
                             'MC_KangaAuto',
                             'MC_KangaAuto',
                             'MC_KangaAuto',
                             'NV',
                             'NV',
                             'NV',   
                            
                             'MC_withGinger',
                             'MC_withGinger',
                             'MC_withGinger',
                             'MC_withKoala',
                             'MC_withKoala',
                             'MC_withKoala',
                             'MC_withVermelho',
                             'MC_withVermelho',
                             'MC_withVermelho',
            
                             'MC_withDodson',
                             'SR_withDodson',
                             'MC_withDodson',
                             'SR_withDodson',
                             'SR_withDodson',
                             'MC_withDodson',
                             'SR_withDodson',
            
                             'MC_withDodson',
                            'SR_withDodson', 
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',            
                            'NV_withDodson',
                            'SR_withDodson',
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',  
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',  
                            'NV_withDodson',
                            'MC_withDodson',
                            'SR_withDodson',  
                          ]
        session_start_times = [ 
                                 0.00,
                                 36.0,
                                 69.5,
                                 0.00,
                                 62.0,
                                 0.00,
                                 89.0,
                                 0.00,
                                 0.00,
                                 0.00,
                                 165.8,
                                 96.0, 
                                 0.00,
                                 0.00,
                                 0.00,
                                 48.0,
                                
                                 59.2,
                                 49.5,
                                 40.0,
                                 50.0,
                                 0.00,
                                 69.8,
                                 85.0,
                                 212.9,
                                 68.5,
            
                                 363,
                                 0.00,
                                 79.0,
                                 162.6,
                                 231.9,
                                 109,
                                 0.00,
            
                                0.00,
                                0.00, 
                                0.00,
                                0.00,
                                0.00,          
                                0.00,
                                93.0,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00,
                                0.00, 
                                0.00,
                                274.4,
                                0.00,
                              ] # in second
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                             'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              ]
        
        animal1_fixedorders = ['dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'ginger','ginger','ginger','koala','koala','koala','vermelho','vermelho',
                               'vermelho','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                              ]
        animal2_fixedorders = ['kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                              ]
        recordedanimals = animal2_fixedorders

        animal1_filenames = ["Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Ginger","Ginger","Ginger", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga",
                              "Kanga","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             
                            ]
        animal2_filenames = ["Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Koala","Koala","Koala","Vermelho","Vermelho",
                             "Vermelho","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                            ]
        
    elif do_OFC:
        # pick only five sessions for each conditions
        neural_record_conditions = [
                                     
                                   ]
        dates_list = [
                      
                     ]
        videodates_list = dates_list
        task_conditions = [
                           
                          ]
        session_start_times = [ 
                                
                              ] # in second
        kilosortvers = [ 

                       ]
    
        animal1_fixedorders = ['dannon']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]
        recordedanimals = animal2_fixedorders
        
        animal1_filenames = ["Dannon"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
    

    
# a test case
if 0: # kanga example
    neural_record_conditions = ['20250415_Kanga_MC_withDodson']
    dates_list = ["20250415"]
    videodates_list = dates_list
    task_conditions = ['MC_withDodson']
    session_start_times = [363] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai9']
    animal1_fixedorders = ['dodson']
    animal2_fixedorders = ['kanga']
    recordedanimals = animal2_fixedorders
    animal1_filenames = ["Dodson"]
    animal2_filenames = ["Kanga"]
if 0: # dodson example 
    neural_record_conditions = ['20250415_Dodson_MC_withKanga']
    dates_list = ["20250415"]
    videodates_list = dates_list
    task_conditions = ['MC_withKanga']
    session_start_times = [363] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai0']
    animal1_fixedorders = ['dodson']
    recordedanimals = animal1_fixedorders
    animal2_fixedorders = ['kanga']
    animal1_filenames = ["Dodson"]
    animal2_filenames = ["Kanga"]
    
ndates = np.shape(dates_list)[0]

session_start_frames = session_start_times * fps # fps is 30Hz

totalsess_time = 600

# video tracking results info
animalnames_videotrack = ['dodson','scorch'] # does not really mean dodson and scorch, instead, indicate animal1 and animal2
bodypartnames_videotrack = ['rightTuft','whiteBlaze','leftTuft','rightEye','leftEye','mouth']


# which camera to analyzed
cameraID = 'camera-2'
cameraID_short = 'cam2'

considerlevertube = 1
considertubeonly = 0

# location of levers and tubes for camera 2
# # camera 1
# lever_locs_camI = {'dodson':np.array([645,600]),'scorch':np.array([425,435])}
# tube_locs_camI  = {'dodson':np.array([1350,630]),'scorch':np.array([555,345])}
# # camera 2
# # location of the estimiated middle of the box
lever_locs_camI = {'dodson':np.array([1325,615]),'scorch':np.array([560,615])}
# # location of the estimated lever
# lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
tube_locs_camI  = {'dodson':np.array([1550,515]),'scorch':np.array([350,515])}
# # old
# # lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
# # tube_locs_camI  = {'dodson':np.array([1650,490]),'scorch':np.array([250,490])}
# # camera 3
# lever_locs_camI = {'dodson':np.array([1580,440]),'scorch':np.array([1296,540])}
# tube_locs_camI  = {'dodson':np.array([1470,375]),'scorch':np.array([805,475])}


if np.shape(session_start_times)[0] != np.shape(dates_list)[0]:
    exit()

    
# define bhv events summarizing variables     
tasktypes_all_dates = np.zeros((ndates,1))
coopthres_all_dates = np.zeros((ndates,1))

succ_rate_all_dates = np.zeros((ndates,1))
interpullintv_all_dates = np.zeros((ndates,1))
trialnum_all_dates = np.zeros((ndates,1))

owgaze1_num_all_dates = np.zeros((ndates,1))
owgaze2_num_all_dates = np.zeros((ndates,1))
mtgaze1_num_all_dates = np.zeros((ndates,1))
mtgaze2_num_all_dates = np.zeros((ndates,1))
pull1_num_all_dates = np.zeros((ndates,1))
pull2_num_all_dates = np.zeros((ndates,1))

ani1_followerPerc_all_dates = np.zeros((ndates,1))
ani2_followerPerc_all_dates = np.zeros((ndates,1))

bhv_intv_all_dates = dict.fromkeys(dates_list, [])

pull_trig_events_all_dates = dict.fromkeys(dates_list, [])
succpull_trig_events_all_dates = dict.fromkeys(dates_list, [])
failpull_trig_events_all_dates = dict.fromkeys(dates_list, [])

pull_infos_all_dates = dict.fromkeys(dates_list, []) # keep some useful information about pulls - time from last reward, number of preceding failed pull etc

spike_trig_events_all_dates = dict.fromkeys(dates_list, [])

bhvevents_aligned_FR_all_dates = dict.fromkeys(dates_list, [])
bhvevents_aligned_FR_allevents_all_dates = dict.fromkeys(dates_list, [])

strategy_aligned_FR_all_dates = dict.fromkeys(dates_list, [])
strategy_aligned_FR_allevents_all_dates = dict.fromkeys(dates_list, [])

# where to save the summarizing data
data_saved_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/3d_recontruction_analysis_self_and_coop_task_data_saved/'

# neural data folder
neural_data_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/'


In [89]:
print(np.shape(neural_record_conditions))
print(np.shape(task_conditions))
print(np.shape(dates_list))
print(np.shape(videodates_list)) 
print(np.shape(session_start_times))

print(np.shape(kilosortvers))

print(np.shape(trig_channelnames))
print(np.shape(animal1_fixedorders)) 
print(np.shape(recordedanimals))
print(np.shape(animal2_fixedorders))

print(np.shape(animal1_filenames))
print(np.shape(animal2_filenames))  

# save the basis information files
data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+\
                       cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
if not os.path.exists(data_saved_subfolder):
    os.makedirs(data_saved_subfolder)

with open(data_saved_subfolder+'/neural_record_conditions_all_dates_'+\
          animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
    pickle.dump(neural_record_conditions, f)
#
with open(data_saved_subfolder+'/task_conditions_all_dates_'+\
          animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
    pickle.dump(task_conditions, f)
#
with open(data_saved_subfolder+'/dates_list_all_dates_'+\
          animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
    pickle.dump(dates_list, f)
#
with open(data_saved_subfolder+'/videodates_list_all_dates_'+\
          animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
    pickle.dump(videodates_list, f)

(67,)
(67,)
(67,)
(67,)
(67,)
(67,)
(67,)
(67,)
(67,)
(67,)
(67,)
(67,)


In [90]:
# basic behavior analysis (define time stamps for each bhv events, etc)

try:
    if redo_anystep:
        dummy
    
    #
    print('loading all data')
    
    # load saved data
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    
    with open(data_saved_subfolder+'/owgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        owgaze1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/owgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        owgaze2_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/mtgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        mtgaze1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/mtgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        mtgaze2_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pull1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pull2_num_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/tasktypes_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        tasktypes_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/coopthres_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        coopthres_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/succ_rate_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        succ_rate_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/interpullintv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        interpullintv_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/trialnum_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        trialnum_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/bhv_intv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhv_intv_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/pull_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pull_trig_events_all_dates = pickle.load(f)    
    with open(data_saved_subfolder+'/succpull_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        succpull_trig_events_all_dates = pickle.load(f)    
    with open(data_saved_subfolder+'/failpull_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        failpull_trig_events_all_dates = pickle.load(f)    
    
    with open(data_saved_subfolder+'/pull_infos_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pull_infos_all_dates  = pickle.load(f)      
    
    with open(data_saved_subfolder+'/ani1_followerPerc_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        ani1_followerPerc_all_dates  = pickle.load(f)
    with open(data_saved_subfolder+'/ani2_followerPerc_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        ani2_followerPerc_all_dates  = pickle.load(f)
    
    # this one should be calculated in the XX_PullStartfocused_xx code
    with open(data_saved_subfolder+'/pull_rts_all_dates_'+doOnsetAfterMin_suffix+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pull_rts_all_dates  = pickle.load(f)
    
    with open(data_saved_subfolder+'/spike_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        spike_trig_events_all_dates = pickle.load(f) 
        
    with open(data_saved_subfolder+'/bhvevents_aligned_FR_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhvevents_aligned_FR_all_dates = pickle.load(f) 
    with open(data_saved_subfolder+'/bhvevents_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhvevents_aligned_FR_allevents_all_dates = pickle.load(f) 
        
    with open(data_saved_subfolder+'/strategy_aligned_FR_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        strategy_aligned_FR_all_dates = pickle.load(f) 
    with open(data_saved_subfolder+'/strategy_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        strategy_aligned_FR_allevents_all_dates = pickle.load(f) 
        
    print('all data from all dates are loaded')

except:

    print('analyze all dates')

    for idate in np.arange(0,ndates,1):
    
        date_tgt = dates_list[idate]
        videodate_tgt = videodates_list[idate]
        
        neural_record_condition = neural_record_conditions[idate]
        
        session_start_time = session_start_times[idate]
        
        kilosortver = kilosortvers[idate]

        trig_channelname = trig_channelnames[idate]
        
        animal1_filename = animal1_filenames[idate]
        animal2_filename = animal2_filenames[idate]
        
        animal1_fixedorder = [animal1_fixedorders[idate]]
        animal2_fixedorder = [animal2_fixedorders[idate]]
        
        recordedanimal = recordedanimals[idate]

        # folder and file path
        camera12_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
        camera23_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"
        
        # 
        try: 
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"                
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
        except:
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
        
        
        # load behavioral results
        try:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_from_task_code/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
            lever_reading_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_lever_reading_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            lever_reading = pd.read_json(lever_reading_json[0])
            # 
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)   
        except:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_from_task_code/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
            lever_reading_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_lever_reading_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            lever_reading = pd.read_json(lever_reading_json[0])
            #
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)

        # get animal info from the session information
        animal1 = session_info['lever1_animal'][0].lower()
        animal2 = session_info['lever2_animal'][0].lower()

        
        # get task type and cooperation threshold
        try:
            coop_thres = session_info["pulltime_thres"][0]
            tasktype = session_info["task_type"][0]
        except:
            coop_thres = 0
            tasktype = 1
        tasktypes_all_dates[idate] = tasktype
        coopthres_all_dates[idate] = coop_thres   

        # successful trial or not
        succtrial_ornot = np.array((trial_record['rewarded']>0).astype(int))
        succpull1_ornot = np.array((np.isin(bhv_data[bhv_data['behavior_events']==1]['trial_number'],trial_record[trial_record['rewarded']>0]['trial_number'])).astype(int))
        succpull2_ornot = np.array((np.isin(bhv_data[bhv_data['behavior_events']==2]['trial_number'],trial_record[trial_record['rewarded']>0]['trial_number'])).astype(int))
        succpulls_ornot = [succpull1_ornot,succpull2_ornot]
            
        # clean up the trial_record
        warnings.filterwarnings('ignore')
        trial_record_clean = pd.DataFrame(columns=trial_record.columns)
        # for itrial in np.arange(0,np.max(trial_record['trial_number']),1):
        for itrial in trial_record['trial_number']:
            # trial_record_clean.loc[itrial] = trial_record[trial_record['trial_number']==itrial+1].iloc[[0]]
            trial_record_clean = trial_record_clean.append(trial_record[trial_record['trial_number']==itrial].iloc[[0]])
        trial_record_clean = trial_record_clean.reset_index(drop = True)

        # change bhv_data time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(bhv_data)[0]),columns=["time_points_new"])
        # for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
        for itrial in np.arange(0,np.shape(trial_record_clean)[0],1):
            # ind = bhv_data["trial_number"]==itrial+1
            ind = bhv_data["trial_number"]==trial_record_clean['trial_number'][itrial]
            new_time_itrial = bhv_data[ind]["time_points"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        bhv_data["time_points"] = time_points_new["time_points_new"]
        bhv_data = bhv_data[bhv_data["time_points"] != 0]


        # analyze behavior results
        # succ_rate_all_dates[idate] = np.sum(trial_record_clean["rewarded"]>0)/np.shape(trial_record_clean)[0]
        succ_rate_all_dates[idate] = np.sum((bhv_data['behavior_events']==3)|(bhv_data['behavior_events']==4))/np.sum((bhv_data['behavior_events']==1)|(bhv_data['behavior_events']==2))
        trialnum_all_dates[idate] = np.shape(trial_record_clean)[0]
        #
        pullid = np.array(bhv_data[(bhv_data['behavior_events']==1) | (bhv_data['behavior_events']==2)]["behavior_events"])
        pulltime = np.array(bhv_data[(bhv_data['behavior_events']==1) | (bhv_data['behavior_events']==2)]["time_points"])
        pullid_diff = np.abs(pullid[1:] - pullid[0:-1])
        pulltime_diff = pulltime[1:] - pulltime[0:-1]
        interpull_intv = pulltime_diff[pullid_diff==1]
        interpull_intv = interpull_intv[interpull_intv<10]
        mean_interpull_intv = np.nanmean(interpull_intv)
        std_interpull_intv = np.nanstd(interpull_intv)
        #
        interpullintv_all_dates[idate] = mean_interpull_intv
        # 
        if np.isin(animal1,animal1_fixedorder):
            pull1_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==1) 
            pull2_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==2)
        else:
            pull1_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==2) 
            pull2_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==1)

        
        # load behavioral event results
        try:
            # dummy
            print('load social gaze with '+cameraID+' only of '+date_tgt)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'rb') as f:
                output_look_ornot = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'rb') as f:
                output_allvectors = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'rb') as f:
                output_allangles = pickle.load(f)  
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'rb') as f:
                output_key_locations = pickle.load(f)
        except:   
            print('analyze social gaze with '+cameraID+' only of '+date_tgt)
            # get social gaze information 
            output_look_ornot, output_allvectors, output_allangles = find_socialgaze_timepoint_singlecam_wholebody(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,
                                                                                                                   considerlevertube,considertubeonly,sqr_thres_tubelever,
                                                                                                                   sqr_thres_face,sqr_thres_body)
            output_key_locations = find_socialgaze_timepoint_singlecam_wholebody_2(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,considerlevertube)
            
            # save data
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'wb') as f:
                pickle.dump(output_look_ornot, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'wb') as f:
                pickle.dump(output_allvectors, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'wb') as f:
                pickle.dump(output_allangles, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'wb') as f:
                pickle.dump(output_key_locations, f)
                

        look_at_other_or_not_merge = output_look_ornot['look_at_other_or_not_merge']
        look_at_tube_or_not_merge = output_look_ornot['look_at_tube_or_not_merge']
        look_at_lever_or_not_merge = output_look_ornot['look_at_lever_or_not_merge']
        look_at_otherlever_or_not_merge = output_look_ornot['look_at_otherlever_or_not_merge']
        look_at_otherface_or_not_merge = output_look_ornot['look_at_otherface_or_not_merge']
        
        # change the unit to second and align to the start of the session
        session_start_time = session_start_times[idate]
        look_at_other_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_other_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_lever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_lever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_tube_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_tube_or_not_merge['dodson'])[0],1)/fps - session_start_time 
        look_at_otherlever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherlever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_otherface_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherface_or_not_merge['dodson'])[0],1)/fps - session_start_time

        
        # find time point of behavioral events
        output_time_points_socialgaze ,output_time_points_levertube = bhv_events_timepoint_singlecam(bhv_data,look_at_other_or_not_merge,look_at_lever_or_not_merge,look_at_tube_or_not_merge)
        time_point_pull1 = output_time_points_socialgaze['time_point_pull1']
        time_point_pull2 = output_time_points_socialgaze['time_point_pull2']
        oneway_gaze1 = output_time_points_socialgaze['oneway_gaze1']
        oneway_gaze2 = output_time_points_socialgaze['oneway_gaze2']
        mutual_gaze1 = output_time_points_socialgaze['mutual_gaze1']
        mutual_gaze2 = output_time_points_socialgaze['mutual_gaze2']
        #
        # newly added, save the output_time_points_socialgaze
        # save the data of output_time_points_socialgaze
        if 0:
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+
                      cameraID+'/'+date_tgt+'/output_time_points_socialgaze.pkl', 'wb') as f:
                pickle.dump(output_time_points_socialgaze, f)
        
        # 
        # mostly just for the sessions in which MC and SR are in the same session 
        firstpulltime = np.nanmin([np.nanmin(time_point_pull1),np.nanmin(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1>(firstpulltime-15)] # 15s before the first pull (animal1 or 2) count as the active period
        oneway_gaze2 = oneway_gaze2[oneway_gaze2>(firstpulltime-15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1>(firstpulltime-15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2>(firstpulltime-15)]  
        #    
        # newly added condition: only consider gaze during the active pulling time (15s after the last pull)    
        lastpulltime = np.nanmax([np.nanmax(time_point_pull1),np.nanmax(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1<(lastpulltime+15)]    
        oneway_gaze2 = oneway_gaze2[oneway_gaze2<(lastpulltime+15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1<(lastpulltime+15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2<(lastpulltime+15)] 
            
        # define successful pulls and failed pulls
        if 0: # old definition; not in use
            trialnum_succ = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']>0])
            bhv_data_succ = bhv_data[np.isin(bhv_data['trial_number'],trialnum_succ)]
            #
            time_point_pull1_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==1]
            time_point_pull2_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==2]
            time_point_pull1_succ = np.round(time_point_pull1_succ,1)
            time_point_pull2_succ = np.round(time_point_pull2_succ,1)
            #
            trialnum_fail = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']==0])
            bhv_data_fail = bhv_data[np.isin(bhv_data['trial_number'],trialnum_fail)]
            #
            time_point_pull1_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==1]
            time_point_pull2_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==2]
            time_point_pull1_fail = np.round(time_point_pull1_fail,1)
            time_point_pull2_fail = np.round(time_point_pull2_fail,1)
        else:
            # a new definition of successful and failed pulls
            # separate successful and failed pulls
            # step 1 all pull and juice
            time_point_pull1 = bhv_data["time_points"][bhv_data["behavior_events"]==1]
            time_point_pull2 = bhv_data["time_points"][bhv_data["behavior_events"]==2]
            time_point_juice1 = bhv_data["time_points"][bhv_data["behavior_events"]==3]
            time_point_juice2 = bhv_data["time_points"][bhv_data["behavior_events"]==4]
            # step 2:
            # pull 1
            # Find the last pull before each juice
            successful_pull1 = [time_point_pull1[time_point_pull1 < juice].max() for juice in time_point_juice1]
            # Convert to Pandas Series
            successful_pull1 = pd.Series(successful_pull1, index=time_point_juice1.index)
            # Find failed pulls (pulls that are not successful)
            failed_pull1 = time_point_pull1[~time_point_pull1.isin(successful_pull1)]
            # pull 2
            # Find the last pull before each juice
            successful_pull2 = [time_point_pull2[time_point_pull2 < juice].max() for juice in time_point_juice2]
            # Convert to Pandas Series
            successful_pull2 = pd.Series(successful_pull2, index=time_point_juice2.index)
            # Find failed pulls (pulls that are not successful)
            failed_pull2 = time_point_pull2[~time_point_pull2.isin(successful_pull2)]
            #
            # step 3:
            time_point_pull1_succ = np.round(successful_pull1,1)
            time_point_pull2_succ = np.round(successful_pull2,1)
            time_point_pull1_fail = np.round(failed_pull1,1)
            time_point_pull2_fail = np.round(failed_pull2,1)
        # 
        time_point_pulls_succfail = { "pull1_succ":time_point_pull1_succ,
                                      "pull2_succ":time_point_pull2_succ,
                                      "pull1_fail":time_point_pull1_fail,
                                      "pull2_fail":time_point_pull2_fail,
                                    }
        
        # 
        # based on time point pull and juice, define some features for each pull action
        pull_infos = get_pull_infos(animal1, animal2, time_point_pull1, time_point_pull2, 
                                    time_point_juice1, time_point_juice2)
        pull_infos_all_dates[date_tgt] = pull_infos

    
            
        # new total session time (instead of 600s) - total time of the video recording
        totalsess_time = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30) 
                
        # # plot behavioral events
        if np.isin(animal1,animal1_fixedorder):
                plot_bhv_events(date_tgt,animal1, animal2, session_start_time, totalsess_time, time_point_pull1, time_point_pull2, oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2)
        else:
                plot_bhv_events(date_tgt,animal2, animal1, session_start_time, totalsess_time, time_point_pull2, time_point_pull1, oneway_gaze2, oneway_gaze1, mutual_gaze2, mutual_gaze1)
        #
        # save behavioral events plot
        if 0:
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            plt.savefig(data_saved_folder+"/bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/'+date_tgt+"_"+cameraID_short+".pdf")

        #
        if np.isin(animal1,animal1_fixedorder):
            owgaze1_num_all_dates[idate] = np.shape(oneway_gaze1)[0]
            owgaze2_num_all_dates[idate] = np.shape(oneway_gaze2)[0]
            mtgaze1_num_all_dates[idate] = np.shape(mutual_gaze1)[0]
            mtgaze2_num_all_dates[idate] = np.shape(mutual_gaze2)[0]
        else:            
            owgaze1_num_all_dates[idate] = np.shape(oneway_gaze2)[0]
            owgaze2_num_all_dates[idate] = np.shape(oneway_gaze1)[0]
            mtgaze1_num_all_dates[idate] = np.shape(mutual_gaze2)[0]
            mtgaze2_num_all_dates[idate] = np.shape(mutual_gaze1)[0]
            
        # 
        # find animal 1 and 2 following percentage
        time_pull1 = np.array(time_point_pull1)
        time_pull2 = np.array(time_point_pull2)
        #
        # animal1's pulls relative to animal2
        following_animal1 = []
        for t_actor in time_pull1:
            diffs = t_actor - time_pull2
            closest_diff = diffs[np.argmin(np.abs(diffs))]
            following_animal1.append(closest_diff > 0)
        frac_following_animal1 = np.sum(following_animal1) / len(following_animal1)
        # 
        # animal2's pulls relative to animal1
        following_animal2 = []
        for t_actor in time_pull2:
            diffs = t_actor - time_pull1
            closest_diff = diffs[np.argmin(np.abs(diffs))]
            following_animal2.append(closest_diff > 0)
        frac_following_animal2 = np.sum(following_animal2) / len(following_animal2)
        #
        if np.isin(animal1,animal1_fixedorder):
            ani1_followerPerc_all_dates[idate] = frac_following_animal1
            ani2_followerPerc_all_dates[idate] = frac_following_animal2
        else:            
            ani1_followerPerc_all_dates[idate] = frac_following_animal2
            ani2_followerPerc_all_dates[idate] = frac_following_animal1
        

            
        #
        # add the pull duration based on the lever-reading; not saved yet
        if 0:
            lever_reading['readout_timepoint_abs'] = np.nan
            trials_id = np.array(trial_record_clean['trial_number'])
            #
            for itrial_id in trials_id:
                #
                ind_ = trial_record_clean['trial_number'] == itrial_id
                trialstarttime_itrial = np.array(trial_record_clean[ind_]['trial_starttime'])[0]
                ind_lever_reading = lever_reading['trial_number'] == itrial_id
                lever_reading.loc[ind_lever_reading, 'readout_timepoint_abs'] = \
                            lever_reading.loc[ind_lever_reading, 'readout_timepoint'] + \
                            trialstarttime_itrial
            #
            lever_reading_lever1 = lever_reading[lever_reading['lever_id']==1]
            lever_reading_lever2 = lever_reading[lever_reading['lever_id']==2]
            #
            if 0:
                lever1_pulltime = np.array(lever_reading_lever1[lever_reading_lever1['pull_or_release']==0]['readout_timepoint_abs'])-\
                        np.array(lever_reading_lever1[lever_reading_lever1['pull_or_release']==1]['readout_timepoint_abs'])
                lever2_pulltime = np.array(lever_reading_lever2[lever_reading_lever2['pull_or_release']==0]['readout_timepoint_abs'])-\
                        np.array(lever_reading_lever2[lever_reading_lever2['pull_or_release']==1]['readout_timepoint_abs'])
            else:
                pulls_lever1  = np.array(lever_reading_lever1[lever_reading_lever1['pull_or_release']==0]['readout_timepoint_abs'])
                release_lever1 = np.array(lever_reading_lever1[lever_reading_lever1['pull_or_release']==1]['readout_timepoint_abs'])
                n_lever1 = min(len(pulls_lever1), len(release_lever1))
                lever1_pulltime = pulls_lever1[:n_lever1] - release_lever1[:n_lever1]

                pulls_lever2  = np.array(lever_reading_lever2[lever_reading_lever2['pull_or_release']==0]['readout_timepoint_abs'])
                release_lever2 = np.array(lever_reading_lever2[lever_reading_lever2['pull_or_release']==1]['readout_timepoint_abs'])
                n_lever2 = min(len(pulls_lever2), len(release_lever2))
                lever2_pulltime = pulls_lever2[:n_lever2] - release_lever2[:n_lever2]
    
    
        
        # analyze the events interval, especially for the pull to other and other to pull interval
        # could be used for define time bin for DBN
        if 0:
            _,_,_,pullTOother_itv, otherTOpull_itv = bhv_events_interval(totalsess_time, session_start_time, time_point_pull1, time_point_pull2, 
                                                                         oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2)
            #
            pull_other_pool_itv = np.concatenate((pullTOother_itv,otherTOpull_itv))
            bhv_intv_all_dates[date_tgt] = {'pull_to_other':pullTOother_itv,'other_to_pull':otherTOpull_itv,
                            'pull_other_pooled': pull_other_pool_itv}
        
        
        # plot key continuous behavioral variables
        if 0:
            filepath_cont_var = data_saved_folder+'bhv_events_continuous_variables_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]+'/'+cameraID+'/'+date_tgt+'/'
            if not os.path.exists(filepath_cont_var):
                os.makedirs(filepath_cont_var)

            savefig = 1
            
            aligntwins = 4 # 5 second
            
            min_length = np.shape(look_at_other_or_not_merge['dodson'])[0] # frame numbers of the video recording

            # NOTE! This one used the wrong and old version of separating successful and failed 
            pull_trig_events_summary, _, _ = plot_continuous_bhv_var_singlecam(filepath_cont_var+date_tgt+cameraID,
                                    aligntwins, savefig, animal1, animal2, 
                                    session_start_time, min_length, succpulls_ornot, time_point_pull1, time_point_pull2, 
                                    oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, animalnames_videotrack,
                                    output_look_ornot, output_allvectors, output_allangles,output_key_locations)
            pull_trig_events_all_dates[date_tgt] = pull_trig_events_summary
            
            # successful pull
            try:
                pull_trig_events_summary, _, _ = plot_continuous_bhv_var_singlecam(filepath_cont_var+date_tgt+cameraID,
                                        aligntwins, savefig, animal1, animal2, 
                                        session_start_time, min_length, succpulls_ornot, 
                                        time_point_pull1_succ, time_point_pull2_succ, 
                                        oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, animalnames_videotrack,
                                        output_look_ornot, output_allvectors, output_allangles,output_key_locations)
                succpull_trig_events_all_dates[date_tgt] = pull_trig_events_summary
            except:
                succpull_trig_events_all_dates[date_tgt] = np.nan
            
            # failed pull
            try:
                pull_trig_events_summary, _, _ = plot_continuous_bhv_var_singlecam(filepath_cont_var+date_tgt+cameraID,
                                        aligntwins, savefig, animal1, animal2, 
                                        session_start_time, min_length, succpulls_ornot, 
                                        time_point_pull1_fail, time_point_pull2_fail, 
                                        oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, animalnames_videotrack,
                                        output_look_ornot, output_allvectors, output_allangles,output_key_locations)
                failpull_trig_events_all_dates[date_tgt] = pull_trig_events_summary
            except:
                failpull_trig_events_all_dates[date_tgt] = np.nan
                
        
        # session starting time compared with the neural recording
        session_start_time_niboard_offset = ni_data['session_t0_offset'] # in the unit of second
        try:
            neural_start_time_niboard_offset = ni_data['trigger_ts'][0]['elapsed_time'] # in the unit of second
        except: # for the multi-animal recording setup
            neural_start_time_niboard_offset = next(
                entry['timepoints'][0]['elapsed_time']
                for entry in ni_data['trigger_ts']
                if entry['channel_name'] == f"{trig_channelname}")
        neural_start_time_session_start_offset = neural_start_time_niboard_offset-session_start_time_niboard_offset
    
    
        # load channel maps
        channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32.mat'
        # channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32_kilosort4_new.mat'
        channel_map_data = scipy.io.loadmat(channel_map_file)
            
        # # load spike sorting results
        if 0:
            print('load spike data for '+neural_record_condition)
            if kilosortver == 2:
                spike_time_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            elif kilosortver == 4:
                spike_time_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            # 
            # align the FR recording time stamps
            spike_time_data = spike_time_data + fs_spikes*neural_start_time_session_start_offset
            # down-sample the spike recording resolution to 30Hz
            spike_time_data = spike_time_data/fs_spikes*fps
            spike_time_data = np.round(spike_time_data)
            #
            if kilosortver == 2:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            elif kilosortver == 4:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            #
            if kilosortver == 2:
                channel_maps_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            elif kilosortver == 4:
                channel_maps_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            #
            if kilosortver == 2:
                channel_pos_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            elif kilosortver == 4:
                channel_pos_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            #
            if kilosortver == 2:
                clusters_info_file = neural_data_folder+neural_record_condition+'/Kilosort/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            elif kilosortver == 4:
                clusters_info_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            #
            # only get the spikes that are manually checked
            try:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['cluster_id'].values
            except:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['id'].values
            #
            clusters_info_data = clusters_info_data[~pd.isnull(clusters_info_data.group)]
            #
            spike_time_data = spike_time_data[np.isin(spike_clusters_data,good_clusters)]
            spike_channels_data = spike_channels_data[np.isin(spike_clusters_data,good_clusters)]
            spike_clusters_data = spike_clusters_data[np.isin(spike_clusters_data,good_clusters)]
            
            #
            nclusters = np.shape(clusters_info_data)[0]
            #
            for icluster in np.arange(0,nclusters,1):
                try:
                    cluster_id = clusters_info_data['id'].iloc[icluster]
                except:
                    cluster_id = clusters_info_data['cluster_id'].iloc[icluster]
                spike_channels_data[np.isin(spike_clusters_data,cluster_id)] = clusters_info_data['ch'].iloc[icluster]   
            # 
            # get the channel to depth information, change 2 shanks to 1 shank 
            try:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1]*2,channel_pos_data[channel_pos_data[:,0]==1,1]*2+1])
                # channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
            except:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
                channel_to_depth[1] = channel_to_depth[1]/30-64 # make the y axis consistent
            #
           
            
            # calculate the firing rate
            # FR_kernel = 0.20 # in the unit of second
            FR_kernel = 1/30 # in the unit of second # 1/30 same resolution as the video recording
            # FR_kernel is sent to to be this if want to explore it's relationship with continuous trackng data
            
            totalsess_time_forFR = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30)  # to match the total time of the video recording
            _,FR_timepoint_allch,FR_allch,FR_zscore_allch = spike_analysis_FR_calculation(fps, FR_kernel, totalsess_time_forFR,
                                                                                          spike_clusters_data, spike_time_data)
            # _,FR_timepoint_allch,FR_allch,FR_zscore_allch = spike_analysis_FR_calculation(fps,FR_kernel,totalsess_time_forFR,
            #                                                                              spike_channels_data, spike_time_data)
            # behavioral events aligned firing rate for each unit
            if 1: 
                print('plot event aligned firing rate')
                #
                savefig = 1
                save_path = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents/"+cameraID+"/"+\
                            animal1_filename+"_"+animal2_filename+'_'+recordedanimal+'Recorded'+"/"+date_tgt
                if not os.path.exists(save_path):
                    os.makedirs(save_path)
                #
                aligntwins = 4 # 5 second
                gaze_thresold = 0.2 # min length threshold to define if a gaze is real gaze or noise, in the unit of second 
                #
                bhvevents_aligned_FR_average_all,bhvevents_aligned_FR_allevents_all = plot_bhv_events_aligned_FR(date_tgt,savefig,save_path, animal1, animal2,time_point_pull1,time_point_pull2,time_point_pulls_succfail,
                                           oneway_gaze1,oneway_gaze2,mutual_gaze1,mutual_gaze2,gaze_thresold,totalsess_time_forFR,
                                           aligntwins,fps,FR_timepoint_allch,FR_zscore_allch,clusters_info_data)
                
                bhvevents_aligned_FR_all_dates[date_tgt] = bhvevents_aligned_FR_average_all
                bhvevents_aligned_FR_allevents_all_dates[date_tgt] = bhvevents_aligned_FR_allevents_all
                
            
            # the three strategy aligned firing rate for each unit
            if 1: 
                print('plot strategy aligned firing rate')
                #
                savefig = 1
                save_path = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents/"+cameraID+"/"+\
                            animal1_filename+"_"+animal2_filename+'_'+recordedanimal+'Recorded'+"/"+date_tgt
                if not os.path.exists(save_path):
                    os.makedirs(save_path)
                #
                stg_twins = 1.5 # 3s, the behavioral event interval used to define strategy, consistent with DBN 3s time lags
                aligntwins = 4 # 5 second
                gaze_thresold = 0.2 # min length threshold to define if a gaze is real gaze or noise, in the unit of second 
                #
                strategy_aligned_FR_average_all,strategy_aligned_FR_allevents_all = plot_strategy_aligned_FR(date_tgt,savefig,save_path, animal1, animal2,time_point_pull1,time_point_pull2,time_point_pulls_succfail,
                                           oneway_gaze1,oneway_gaze2,mutual_gaze1,mutual_gaze2,gaze_thresold,totalsess_time_forFR,
                                           aligntwins,stg_twins,fps,FR_timepoint_allch,FR_zscore_allch,clusters_info_data)
                
                strategy_aligned_FR_all_dates[date_tgt] = strategy_aligned_FR_average_all
                strategy_aligned_FR_allevents_all_dates[date_tgt] = strategy_aligned_FR_allevents_all
                
            
            #
            # Run PCA analysis
            FR_zscore_allch_np_merged = np.array(pd.DataFrame(FR_zscore_allch).T)
            FR_zscore_allch_np_merged = FR_zscore_allch_np_merged[~np.isnan(np.sum(FR_zscore_allch_np_merged,axis=1)),:]
            # # run PCA on the entire session
            pca = PCA(n_components=3)
            FR_zscore_allch_PCs = pca.fit_transform(FR_zscore_allch_np_merged.T)
            #
            # # run PCA around the -PCAtwins to PCAtwins for each behavioral events
            PCAtwins = 4 # 5 second
            gaze_thresold = 0.5 # min length threshold to define if a gaze is real gaze or noise, in the unit of second 
            savefigs = 0 
            if 0:
                PCA_around_bhv_events(FR_timepoint_allch,FR_zscore_allch_np_merged,time_point_pull1,time_point_pull2,time_point_pulls_succfail, 
                              oneway_gaze1,oneway_gaze2,mutual_gaze1,mutual_gaze2,gaze_thresold,totalsess_time_forFR,PCAtwins,fps,
                              savefigs,data_saved_folder,cameraID,animal1_filename,animal2_filename,date_tgt)
            if 0:
                if (np.isin(animal1, ['dodson'])) | (np.isin(animal2, ['kanga'])):
                    PCA_around_bhv_events_video(FR_timepoint_allch,FR_zscore_allch_np_merged,time_point_pull1,time_point_pull2,time_point_pulls_succfail, 
                                      oneway_gaze1,oneway_gaze2,mutual_gaze1,mutual_gaze2,gaze_thresold,totalsess_time_forFR,PCAtwins,fps,
                                      data_saved_folder,cameraID,animal1_filename,animal2_filename,date_tgt)
                elif (np.isin(animal2, ['dodson'])) | (np.isin(animal1, ['kanga'])):
                    time_point_pulls_succfail_rev = time_point_pulls_succfail.copy()
                    time_point_pulls_succfail_rev['pull1_succ'] = time_point_pulls_succfail['pull2_succ']
                    time_point_pulls_succfail_rev['pull1_fail'] = time_point_pulls_succfail['pull2_fail']
                    time_point_pulls_succfail_rev['pull2_succ'] = time_point_pulls_succfail['pull1_succ']
                    time_point_pulls_succfail_rev['pull2_fail'] = time_point_pulls_succfail['pull1_fail']
                    PCA_around_bhv_events_video(FR_timepoint_allch,FR_zscore_allch_np_merged,time_point_pull2,time_point_pull1,time_point_pulls_succfail_rev, 
                                      oneway_gaze2,oneway_gaze1,mutual_gaze2,mutual_gaze1,gaze_thresold,totalsess_time_forFR,PCAtwins,fps,
                                      data_saved_folder,cameraID,animal1_filename,animal2_filename,date_tgt)
            
            
            
            # do the spike triggered average of different bhv variables, for the single camera tracking, look at the pulling and social gaze actions
            # the goal is to get a sense for glm
            if 1: 
                print('plot spike triggered bhv variables')

                savefig = 1
                save_path = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents/"+cameraID+"/"+\
                            animal1_filename+"_"+animal2_filename+'_'+recordedanimal+'Recorded'+"/"+date_tgt
                if not os.path.exists(save_path):
                    os.makedirs(save_path)
                #
                do_shuffle = 0
                #
                min_length = np.shape(look_at_other_or_not_merge['dodson'])[0] # frame numbers of the video recording
                #
                trig_twins = [-4,4] # the time window to examine the spike triggered average, in the unit of s
                
                gaze_thresold = 0.2
                
                stg_twins = 3 # 3s, the behavioral event interval used to define strategy, consistent with DBN 3s time lags
                #
                spike_trig_average_all =  plot_spike_triggered_singlecam_bhvevent(date_tgt,savefig,save_path, animal1, animal2, session_start_time,min_length, trig_twins,
                                                                              stg_twins, time_point_pull1, time_point_pull2, time_point_pulls_succfail,
                                                                              oneway_gaze1,oneway_gaze2,mutual_gaze1,mutual_gaze2,gaze_thresold,animalnames_videotrack,
                                                                              spike_clusters_data, spike_time_data,spike_channels_data,do_shuffle)

                spike_trig_events_all_dates[date_tgt] = spike_trig_average_all

            
        # load filtered lfp
        if 0:
            print('load LFP data for '+neural_record_condition)
            lfp_filt_filename = neural_data_folder+neural_record_condition+'/lfp_filt_subsample.txt' # already downsample to 30Hz
            lfp_filt_data_df = genfromtxt(lfp_filt_filename, delimiter=',')
            # aligned to the session start
            lfp_filt_sess_aligned=lfp_filt_data_df[:,int(-neural_start_time_session_start_offset*30):]
            # normalize the activity to 0 - 1
            lfp_filt_sess_aligned = (lfp_filt_sess_aligned-np.min(lfp_filt_sess_aligned))/(np.max(lfp_filt_sess_aligned)-np.min(lfp_filt_sess_aligned))

        
        # plot the tracking demo video
        if 0: 
            print('make the demo videos')
            if 0:
                # all the bhv traces in the same panel
                tracking_video_singlecam_wholebody_withNeuron_demo(bodyparts_locs_camI,output_look_ornot,output_allvectors,output_allangles,
                                                  lever_locs_camI,tube_locs_camI,time_point_pull1,time_point_pull2,
                                                  animalnames_videotrack,bodypartnames_videotrack,date_tgt,
                                                  animal1_filename,animal2_filename,session_start_time,fps,nframes,cameraID,
                                                  video_file_original,sqr_thres_tubelever,sqr_thres_face,sqr_thres_body,
                                                  spike_time_data,lfp_filt_sess_aligned,spike_channels_data,channel_to_depth)
            if 1:
                # all the bhv traces are in separate panels
                tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo(bodyparts_locs_camI,output_look_ornot,output_allvectors,output_allangles,
                                                 lever_locs_camI,tube_locs_camI,time_point_pull1,time_point_pull2,
                                                 animalnames_videotrack,bodypartnames_videotrack,date_tgt,
                                                 animal1_filename,animal2_filename,session_start_time,fps,nframes,cameraID,
                                                 video_file_original,sqr_thres_tubelever,sqr_thres_face,sqr_thres_body,
                                                 spike_time_data,lfp_filt_sess_aligned,spike_channels_data,channel_to_depth)
        
        # plot the example frame from the tracking demo video
        if 0: 
            print('print the example frame from the demo videos')
            if 1:
                example_frame = 60*30+1
                start_frame = 55*30
                # all the bhv traces are in separate panels
                tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo(bodyparts_locs_camI,output_look_ornot,output_allvectors,output_allangles,
                                                 lever_locs_camI,tube_locs_camI,time_point_pull1,time_point_pull2,
                                                 animalnames_videotrack,bodypartnames_videotrack,date_tgt,
                                                 animal1_filename,animal2_filename,session_start_time,fps,start_frame,example_frame,cameraID,
                                                 video_file_original,sqr_thres_tubelever,sqr_thres_face,sqr_thres_body,
                                                 spike_time_data,lfp_filt_sess_aligned,spike_channels_data,channel_to_depth)
                savefig = 1
                save_path = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents/"+cameraID+"/"+\
                            animal1_filename+"_"+animal2_filename+'_'+recordedanimal+'Recorded'+"/"+date_tgt+"/"
                if not os.path.exists(save_path):
                    os.makedirs(save_path)
                if savefig:
                    plt.savefig(save_path+'singlecam_wholebody_tracking_withNeuron_sepbhv_demo_oneframe.pdf')
        

    # save data
    if 0:
        
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
                
        # with open(data_saved_subfolder+'/DBN_input_data_alltypes_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
        #     pickle.dump(DBN_input_data_alltypes, f)

        with open(data_saved_subfolder+'/owgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(owgaze1_num_all_dates, f)
        with open(data_saved_subfolder+'/owgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(owgaze2_num_all_dates, f)
        with open(data_saved_subfolder+'/mtgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(mtgaze1_num_all_dates, f)
        with open(data_saved_subfolder+'/mtgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(mtgaze2_num_all_dates, f)
        with open(data_saved_subfolder+'/pull1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pull1_num_all_dates, f)
        with open(data_saved_subfolder+'/pull2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pull2_num_all_dates, f)

        with open(data_saved_subfolder+'/tasktypes_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(tasktypes_all_dates, f)
        with open(data_saved_subfolder+'/coopthres_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(coopthres_all_dates, f)
        with open(data_saved_subfolder+'/succ_rate_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(succ_rate_all_dates, f)
        with open(data_saved_subfolder+'/interpullintv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(interpullintv_all_dates, f)
        with open(data_saved_subfolder+'/trialnum_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(trialnum_all_dates, f)
        with open(data_saved_subfolder+'/bhv_intv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhv_intv_all_dates, f)
            
        with open(data_saved_subfolder+'/pull_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pull_trig_events_all_dates, f) 
        with open(data_saved_subfolder+'/succpull_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(succpull_trig_events_all_dates, f) 
        with open(data_saved_subfolder+'/failpull_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(failpull_trig_events_all_dates, f) 
            
        with open(data_saved_subfolder+'/pull_infos_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pull_infos_all_dates, f)             
            
        with open(data_saved_subfolder+'/spike_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(spike_trig_events_all_dates, f)  
    
        with open(data_saved_subfolder+'/bhvevents_aligned_FR_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhvevents_aligned_FR_all_dates, f) 
        with open(data_saved_subfolder+'/bhvevents_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhvevents_aligned_FR_allevents_all_dates, f) 
            
        with open(data_saved_subfolder+'/strategy_aligned_FR_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(strategy_aligned_FR_all_dates, f) 
        with open(data_saved_subfolder+'/strategy_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(strategy_aligned_FR_allevents_all_dates, f) 
    
    
    # only save a subset 
    if 0:
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
    
        # animal1's percentage of following
        with open(data_saved_subfolder+'/ani1_followerPerc_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(ani1_followerPerc_all_dates, f)
        # animal2's percentage of following
        with open(data_saved_subfolder+'/ani2_followerPerc_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(ani2_followerPerc_all_dates, f) 

   
    

loading all data
all data from all dates are loaded


In [91]:
# get the follower/leader definition based on Jing's method

# ── Load follower/leader definition TSV ──────────────────────────────────────
# df_follower = pd.read_csv('Jings_FollowerLeader_definition_old_1s.tsv', sep='\t')
df_follower = pd.read_csv('Jings_FollowerLeader_definition.tsv', sep='\t')

# ── Remove ambiguous sessions where both follower scores are zero ─────────────
ambiguous = (df_follower['followerscore1'] == 0) & (df_follower['followerscore2'] == 0)
print(f"Ambiguous sessions (both scores == 0): {ambiguous.sum()}")
# print(df_follower[ambiguous][['date', 'recorded_animal', 'followerscore1', 'followerscore2', 'FOLLOWER']])

# set FOLLOWER to NaN for ambiguous sessions
# df_follower.loc[ambiguous, 'FOLLOWER'] = np.nan

# Normalize for matching: lowercase date, lowercase animal names
df_follower['date_str'] = df_follower['date'].astype(str).str.strip()
df_follower['recorded_animal_lower'] = df_follower['recorded_animal'].str.lower().str.strip()
df_follower['FOLLOWER_lower'] = df_follower['FOLLOWER'].str.lower().str.strip()

Ambiguous sessions (both scores == 0): 22


In [92]:
# df_follower

### look at the neural pc space and explore it's relationship with interpull interval
#### to check Jing's hypothesis, I will load both recorded animals and combine them together

In [93]:
# helper functions

import pandas as pd
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter1d
from scipy.spatial import ConvexHull

def smooth_trajectory(traj, sigma=2):
    """Gaussian smooth each PC dimension independently."""
    return np.stack([gaussian_filter1d(traj[:, i], sigma=sigma) for i in range(traj.shape[1])], axis=1)

def mean_curvature(traj):
    """Mean turning angle between consecutive steps (in radians)."""
    steps = np.diff(traj, axis=0)  # (T-1, 3)
    norms = np.linalg.norm(steps, axis=1, keepdims=True)
    # avoid division by zero
    valid = (norms[:, 0] > 0)
    steps_normed = np.where(norms > 0, steps / np.where(norms > 0, norms, 1), 0)
    # angle between consecutive unit vectors
    dots = np.sum(steps_normed[:-1] * steps_normed[1:], axis=1)
    dots = np.clip(dots, -1, 1)
    angles = np.arccos(dots)
    valid_angles = angles[valid[:-1] & valid[1:]]
    return np.mean(valid_angles) if len(valid_angles) > 0 else np.nan

def radius_of_gyration(traj):
    if 0:
        """RMS distance of trajectory points from their centroid."""
        centroid = np.mean(traj, axis=0)
        return np.sqrt(np.mean(np.sum((traj - centroid)**2, axis=1)))
    if 1:
        """mean instead of RMS — closer to circle_radius_2d but in 3D"""
        centroid = np.mean(traj, axis=0)
        return np.mean(np.linalg.norm(traj - centroid, axis=1))

def convex_hull_volume(traj):
    """Convex hull volume in 3D PC space. Needs >= 4 non-coplanar points."""
    try:
        hull = ConvexHull(traj)
        return hull.volume
    except Exception:
        return np.nan
    
def angle_span_net(traj):
    """
    Net angular displacement of trajectory around its centroid,
    projected onto the trajectory's own dominant 2D plane via PCA.
    Analogous to Jing's diameter — how much of the circle is swept.
    Returns net absolute angle (radians).
    """
    centroid = np.mean(traj, axis=0)
    centered = traj - centroid
    try:
        pca_traj  = PCA(n_components=2)
        projected = pca_traj.fit_transform(centered)
        angles    = np.arctan2(projected[:, 1], projected[:, 0])
        angles_unwrapped = np.unwrap(angles)
        return np.abs(angles_unwrapped[-1] - angles_unwrapped[0])
    except Exception:
        return np.nan
    
def angle_span_total(traj):
    """
    Total angular path length of trajectory around its centroid,
    projected onto the trajectory's own dominant 2D plane via PCA.
    Sum of all absolute angular steps — more like tortuosity in angular space.
    Returns total angular path length (radians).
    """
    centroid = np.mean(traj, axis=0)
    centered = traj - centroid
    try:
        pca_traj  = PCA(n_components=2)
        projected = pca_traj.fit_transform(centered)
        angles    = np.arctan2(projected[:, 1], projected[:, 0])
        angles_unwrapped = np.unwrap(angles)
        angle_steps = np.abs(np.diff(angles_unwrapped))
        return np.sum(angle_steps)
    except Exception:
        return np.nan
    
def angle_span_net_3d(traj):
    """
    Net angular displacement in 3D, measured as the angle between
    the first and last position vectors relative to the centroid.
    """
    centroid = np.mean(traj, axis=0)
    centered = traj - centroid
    
    v_start = centered[0]
    v_end   = centered[-1]
    
    norm_start = np.linalg.norm(v_start)
    norm_end   = np.linalg.norm(v_end)
    
    if norm_start < 1e-10 or norm_end < 1e-10:
        return np.nan
    
    cos_angle = np.dot(v_start, v_end) / (norm_start * norm_end)
    cos_angle = np.clip(cos_angle, -1, 1)
    return np.arccos(cos_angle)  # radians

def angle_span_total_3d(traj):
    """
    Total angular path length in 3D.
    Sum of angles between consecutive position vectors relative to centroid.
    """
    centroid = np.mean(traj, axis=0)
    centered = traj - centroid
    
    total_angle = 0.0
    valid_steps = 0
    
    for i in range(len(centered) - 1):
        v1 = centered[i]
        v2 = centered[i + 1]
        
        n1 = np.linalg.norm(v1)
        n2 = np.linalg.norm(v2)
        
        if n1 < 1e-10 or n2 < 1e-10:
            continue
        
        cos_angle = np.dot(v1, v2) / (n1 * n2)
        cos_angle = np.clip(cos_angle, -1, 1)
        total_angle += np.arccos(cos_angle)
        valid_steps += 1
    
    return total_angle if valid_steps > 0 else np.nan

In [97]:
# do the calculation
    
#
recordedanimals = ['kanga','dodson']
animal1_fixedorders_tgt = ['dannon','dodson'] # for loading file purpose
animal2_fixedorders_tgt = ['kanga','ginger'] # for loading file purpose

savefile_sufix_tgt = '_DLPFCs'

nrecordedanimals = np.shape(recordedanimals)[0]

tgt_task_condition = 'MC'

# initialize master dataframe
df_pca_features = []  # must be a plain list, not a DataFrame

for irecordedanimal in np.arange(0, nrecordedanimals, 1):

    recordedanimal = recordedanimals[irecordedanimal]
    animal1_fixedorder_tgt = animal1_fixedorders_tgt[irecordedanimal]
    animal2_fixedorder_tgt = animal2_fixedorders_tgt[irecordedanimal]

    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix_tgt+\
        '/'+cameraID+'/'+animal1_fixedorder_tgt+animal2_fixedorder_tgt+'/'

    
    with open(data_saved_subfolder+'/succ_rate_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        succ_rate_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/dates_list_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        dates_list_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/task_conditions_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        task_conditions_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull_infos_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        pull_infos_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull_rts_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        pull_rts_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/bhvevents_aligned_FR_allevents_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        bhvevents_aligned_FR_allevents_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/ani1_followerPerc_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        ani1_followerPerc_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/ani2_followerPerc_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        ani2_followerPerc_all_dates = pickle.load(f)

    if recordedanimal == animal1_fixedorder_tgt:
        tgtanimal_followerPer_all_dates = ani1_followerPerc_all_dates
        # followerPec ratio
        selfvspartner_followerPer_all_dates = ani1_followerPerc_all_dates/ani2_followerPerc_all_dates
        #
    elif recordedanimal == animal2_fixedorder_tgt:
        tgtanimal_followerPer_all_dates = ani2_followerPerc_all_dates
        # followerPec ratio
        selfvspartner_followerPer_all_dates = ani2_followerPerc_all_dates/ani1_followerPerc_all_dates

    with open(data_saved_subfolder+'/pull1_num_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        pull1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull2_num_all_dates_'+\
              animal1_fixedorder_tgt+animal2_fixedorder_tgt+'.pkl', 'rb') as f:
        pull2_num_all_dates = pickle.load(f)

    if recordedanimal == animal1_fixedorder_tgt:
        selfvspartner_pullnum_all_dates = pull1_num_all_dates / pull2_num_all_dates
        self_pullnum_all_dates = pull1_num_all_dates
        partner_pullnum_all_dates = pull2_num_all_dates
    #
    elif recordedanimal == animal2_fixedorder_tgt:
        selfvspartner_pullnum_all_dates = pull2_num_all_dates / pull1_num_all_dates
        self_pullnum_all_dates = pull2_num_all_dates
        partner_pullnum_all_dates = pull1_num_all_dates

    tgt_indices = [i for i, cond in enumerate(task_conditions_all_dates) if tgt_task_condition in cond]
    dates_tgt = [dates_list_all_dates[i] for i in tgt_indices]
    ndates = len(dates_tgt)

    task_conditions_tgt      = [task_conditions_all_dates[i]        for i in tgt_indices]
    succ_rate_tgt      = [succ_rate_all_dates[i]        for i in tgt_indices]
    tgtanimal_followerPer_tgt = [tgtanimal_followerPer_all_dates[i]  for i in tgt_indices]
    selfvspartner_pullnum_tgt = [selfvspartner_pullnum_all_dates[i]  for i in tgt_indices]
    partner_pullnum_tgt = [partner_pullnum_all_dates[i]  for i in tgt_indices]
    self_pullnum_tgt = [self_pullnum_all_dates[i]  for i in tgt_indices]
    selfvspartner_followerPer_tgt = [selfvspartner_followerPer_all_dates[i]  for i in tgt_indices]
    

    for idate in np.arange(0, ndates, 1):

        date_tgt = dates_tgt[idate]

        ipi_allpulls_idate = np.array(pull_rts_all_dates[date_tgt][recordedanimal])

        FR_pullaligned_allneuron = bhvevents_aligned_FR_allevents_all_dates[date_tgt][recordedanimal+' pull']
        neuron_ids = list(FR_pullaligned_allneuron.keys())
        nneurons   = len(neuron_ids)

        if nneurons < 5:
            print(f'too few neurons: {date_tgt} {recordedanimal}')
            continue

        FR_stack = np.array([FR_pullaligned_allneuron[nid]['FR_allevents'] for nid in neuron_ids])

        npulls      = FR_stack.shape[2]
        nTimePoints = FR_stack.shape[1]

        if len(ipi_allpulls_idate) != npulls:
            print(f"Pull mismatch: {date_tgt} {recordedanimal} — skipping")
            continue

        FR_concat = FR_stack.transpose(1, 2, 0).reshape(npulls * nTimePoints, nneurons)

        nan_frac = np.sum(np.isnan(FR_concat)) / FR_concat.size
        if nan_frac > 0.05:
            print(f"Too many NaNs: {date_tgt} {recordedanimal} — skipping")
            continue

        FR_concat = np.nan_to_num(FR_concat, nan=0.0)

        pca = PCA(n_components=3)
        FR_pca = pca.fit_transform(FR_concat)  # (npulls*nTimePoints, 3)

        FR_pca_perpull = [FR_pca[ipull * nTimePoints:(ipull + 1) * nTimePoints, :]
                          for ipull in range(npulls)]

        
        # ── look up follower/leader role ──────────────────────────────────
        date_str = str(date_tgt).strip()
        match = df_follower[
            (df_follower['date_str'] == date_str) &
            (df_follower['recorded_animal_lower'] == recordedanimal.lower().strip())
        ]
        if len(match) == 0:
            role = np.nan
        else:
            follower_val = match.iloc[0]['FOLLOWER_lower']
            if pd.isna(follower_val):
                role = np.nan
            elif follower_val == recordedanimal.lower().strip():
                role = 'follower'
            else:
                role = 'leader'
        
        
        for ipull in range(npulls):

            traj = FR_pca_perpull[ipull]         # (nTimePoints, 3)

            # ── smooth ───────────────────────────────────────────────────────
            traj = smooth_trajectory(traj, sigma=10)
            
            # ── only prepull ──────────────────────────────────────────────
            traj = traj[:int(nTimePoints / 2), :] # first half only (-4s to 0s)

            # ── existing features ────────────────────────────────────────────
            steps        = np.diff(traj, axis=0)
            step_lengths = np.linalg.norm(steps, axis=1)

            path_length      = np.sum(step_lengths)
            net_displacement = np.linalg.norm(traj[-1] - traj[0])
            tortuosity       = path_length / net_displacement if net_displacement > 0 else np.nan
            mean_speed_val   = np.mean(step_lengths)

            # ── new features ─────────────────────────────────────────────────
            mean_curv    = mean_curvature(traj)
            rog          = radius_of_gyration(traj)
            chull_vol    = convex_hull_volume(traj)
            # angle_span_net_val   = angle_span_net(traj)
            # angle_span_total_val = angle_span_total(traj)
            # angle_span_net_val   = angle_span_net_3d(traj)
            # angle_span_total_val = angle_span_total_3d(traj)
            angle_span_net_val   = angle_span_net_3d(traj) / path_length
            angle_span_total_val = angle_span_total_3d(traj) / path_length
            
            

            df_pca_features.append({
                'date':             date_tgt,
                'recorded_animal':  recordedanimal,
                'task_condition':   task_conditions_tgt[idate],
                'succ_rate':   succ_rate_tgt[idate][0],
                'neuron_num': nneurons,
                'role':             role,                          # ← new column
                'followerPercentage':        tgtanimal_followerPer_tgt[idate][0],
                'selfvspartner_pullnum':     selfvspartner_pullnum_tgt[idate][0],
                'self_pullnum':      self_pullnum_tgt[idate][0],
                'partner_pullnum':      partner_pullnum_tgt[idate][0],
                'selfvspartner_followerPer': selfvspartner_followerPer_tgt[idate][0],
                'pull_id':          ipull,
                'ipi':              ipi_allpulls_idate[ipull],
                'path_length':      path_length,
                'net_displacement': net_displacement,
                'tortuosity':       tortuosity,
                'mean_speed':       mean_speed_val,
                'mean_curvature':   mean_curv,
                'radius_of_gyration': rog,
                'convex_hull_volume': chull_vol,
                'angle_span_net':      angle_span_net_val,
                'angle_span_total':    angle_span_total_val,
                'pca_var_explained_pc1': pca.explained_variance_ratio_[0],
                'pca_var_explained_pc2': pca.explained_variance_ratio_[1],
                'pca_var_explained_pc3': pca.explained_variance_ratio_[2],
            })

# convert to dataframe
df_pca_features = pd.DataFrame(df_pca_features)


/tmp/ipykernel_928730/865208172.py:70: RuntimeWarning: invalid value encountered in true_divide
  selfvspartner_pullnum_all_dates = pull1_num_all_dates / pull2_num_all_dates


too few neurons: 20240610_MC dodson
too few neurons: 20240719 dodson
too few neurons: 20250424_MC dodson
too few neurons: 20250430_MC dodson


In [109]:
# df_pca_features

In [117]:
valid_conditions = {
    'dodson': ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': [ 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': ['MC', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': ['MC_withKanga', ],
    
    'kanga':  ['MC', 'MC_withDodson','MC_withGinger', 'MC_withKoala', 'MC_withVermelho'],
    # 'kanga':  ['MC', 'MC_withDodson', 'MC_withVermelho'],
    # 'kanga':  ['MC_withDodson',],
    # 'kanga':  ['MC', 'MC_withDodson', ],
}

# filter each animal by their valid task conditions
df_filtered_full = pd.concat([
    df_pca_features[(df_pca_features['recorded_animal']==animal) &
                    (df_pca_features['task_condition'].isin(conds))]
    for animal, conds in valid_conditions.items()
])

df_filtered_full.to_csv("neuroPCA_features_alldates_singletrials.csv", index=False)

# df_filtered_full


In [99]:
from scipy import stats

def safe_pearsonr(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 10: # trial/pull number 
        return np.nan
    # return stats.pearsonr(x[mask], y[mask])[0]
    return stats.spearmanr(x[mask], y[mask])[0]

df_corr_results = []
for (date, animal), group in df_pca_features.groupby(['date', 'recorded_animal']):

    ipi = group['ipi'].values
    Q1, Q3 = np.percentile(ipi[np.isfinite(ipi)], [25, 75])
    IQR = Q3 - Q1
    ipi_mask = (ipi >= Q1 - 1.5 * IQR) & (ipi <= Q3 + 1.5 * IQR)
    group = group[ipi_mask]

    if len(group) < 10:
        continue

    ipi = group['ipi'].values

    df_corr_results.append({
        'date':             date,
        'recorded_animal':  animal,
        'task_condition':   group['task_condition'].iloc[0],
        'succ_rate':   group['succ_rate'].iloc[0],
        'neuron_num':   group['neuron_num'].iloc[0],
        'follower_percentage':     group['followerPercentage'].iloc[0],
        'role':             group['role'].iloc[0],                    # ← new
        'selfvspartner_pullnum':   group['selfvspartner_pullnum'].iloc[0],
        'self_pullnum':   group['self_pullnum'].iloc[0],
        'partner_pullnum':   group['partner_pullnum'].iloc[0],
        'selfvspartner_followerPer':   group['selfvspartner_followerPer'].iloc[0],
        'r_pathlength':        safe_pearsonr(ipi, group['path_length'].values),
        'r_netdisplacement':   safe_pearsonr(ipi, group['net_displacement'].values),
        'r_tortuosity':        safe_pearsonr(ipi, group['tortuosity'].values),
        'r_meanspeed':         safe_pearsonr(ipi, group['mean_speed'].values),
        'r_meancurvature':     safe_pearsonr(ipi, group['mean_curvature'].values),
        'r_radiusofgyration':  safe_pearsonr(ipi, group['radius_of_gyration'].values),
        'r_convexhullvolume':  safe_pearsonr(ipi, group['convex_hull_volume'].values),
        'r_anglespannet':            safe_pearsonr(ipi, group['angle_span_net'].values),
        'r_anglespantotal':          safe_pearsonr(ipi, group['angle_span_total'].values),
    
    })

df_corr_results = pd.DataFrame(df_corr_results)

# ── fill NaN roles using follower_percentage as fallback ─────────────────
if 0:
    mask_nan_role = df_corr_results['role'].isna()
    df_corr_results.loc[mask_nan_role, 'role'] = df_corr_results.loc[mask_nan_role, 'follower_percentage'].apply(
        lambda x: 'follower' if x > 0.5 else 'leader' if pd.notna(x) else np.nan
    )
    


In [107]:
valid_conditions = {
    'dodson': ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': [ 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': ['MC', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': ['MC_withKanga', ],
    
    'kanga':  ['MC', 'MC_withDodson','MC_withGinger', 'MC_withKoala', 'MC_withVermelho'],
    # 'kanga':  ['MC', 'MC_withDodson', 'MC_withVermelho'],
    # 'kanga':  ['MC_withDodson',],
    # 'kanga':  ['MC', 'MC_withDodson', ],
}

# filter each animal by their valid task conditions
df_filtered = pd.concat([
    df_corr_results[(df_corr_results['recorded_animal']==animal) &
                    (df_corr_results['task_condition'].isin(conds)) & 
                    (df_corr_results['succ_rate']>0.2)]
    for animal, conds in valid_conditions.items()
])

df_filtered.to_csv("neuroPCA_features_corr_with_RT_alldates.csv", index=False)

# df_filtered

### add some helper function and do the plotting

In [ ]:
from statsmodels.formula.api import mixedlm
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
from scipy import stats

def cohens_d(a, b):
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else np.nan

def pval_to_stars(p):
    if p < 0.001:  return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else:          return 'ns'

def lmm_test(df_subset, measure_col, split_col, threshold):
    """
    Two models to control for animal identity:
    
    Model 1 - LMM: measure ~ group, random intercept per animal
        Tests if group effect holds after accounting for between-animal variance
    
    Model 2 - fixed effect control: measure ~ group + animal (OLS)
        Tests if group effect survives when animal identity is explicitly partialled out
        More interpretable with only 2 animals
    """
    df_lmm = df_subset[[measure_col, split_col, 'recorded_animal']].dropna().copy()
    df_lmm['group'] = (df_lmm[split_col] > threshold).astype(int)

    # Model 1: LMM with random intercept per animal
    # group effect p-value after soaking up between-animal variance
    try:
        md   = mixedlm(f"{measure_col} ~ group", df_lmm, groups=df_lmm['recorded_animal'])
        mdf  = md.fit(reml=True, method='lbfgs')
        pval_lmm = mdf.pvalues['group']
    except Exception:
        pval_lmm = np.nan

    # Model 2: OLS with animal as fixed covariate
    # most transparent with only 2 animals — directly removes animal mean difference
    try:
        from statsmodels.formula.api import ols
        md_ols  = ols(f"{measure_col} ~ group + recorded_animal", df_lmm)
        mdf_ols = md_ols.fit()
        pval_ols = mdf_ols.pvalues['group']
    except Exception:
        pval_ols = np.nan

    return pval_lmm, pval_ols


def stats_label(a, b, df_subset=None, measure_col=None, split_col=None, threshold=None):
    a, b = np.array(a), np.array(b)
    d = cohens_d(a, b)
    _, pval_welch = stats.ttest_ind(a, b, equal_var=False)
    _, pval_mw    = stats.mannwhitneyu(a, b, alternative='two-sided')

    label = (f"Cohen's d={d:.3f}\n"
             f"Welch p={pval_welch:.3f}{pval_to_stars(pval_welch)} | "
             f"MW p={pval_mw:.3f}{pval_to_stars(pval_mw)}\n"
             f"n=({len(a)},{len(b)})")

    if df_subset is not None:
        pval_lmm, pval_ols = lmm_test(df_subset, measure_col, split_col, threshold)
        label += (f"\nLMM(random animal) p={pval_lmm:.3f}{pval_to_stars(pval_lmm)} | "
                  f"OLS(fixed animal) p={pval_ols:.3f}{pval_to_stars(pval_ols)}")

    return label

def plot_violin_with_points(ax, data_list, labels, title, ylabel, jitter=0.05, point_colors=None):
    ax.violinplot(data_list, positions=range(len(data_list)), showmedians=True)
    for i, data in enumerate(data_list):
        x = np.random.uniform(i - jitter, i + jitter, size=len(data))
        colors = point_colors[i] if point_colors is not None else ['black'] * len(data)
        ax.scatter(x, data, alpha=0.6, s=20, zorder=3, c=colors)
        # test against 0
        clean = data.dropna() if hasattr(data, 'dropna') else data[np.isfinite(data)]
        if len(clean) >= 3:
            _, pval = stats.ttest_1samp(clean, popmean=0)
            star  = pval_to_stars(pval)
            y_pos = np.max(clean) + 0.02 * (np.max(clean) - np.min(clean))
            ax.text(i, y_pos, f'{star}\np={pval:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

def get_threshold(series, fixed_val, mode):
    return series.median() if mode == 'median' else fixed_val

def get_point_colors(df_subset, measure_col, split_col, threshold):
    low  = df_subset[df_subset[split_col] <= threshold]
    high = df_subset[df_subset[split_col] >  threshold]
    colors_low  = [animal_colors.get(a, 'black') for a in low['recorded_animal']]
    colors_high = [animal_colors.get(a, 'black') for a in high['recorded_animal']]
    return [colors_low, colors_high]

animal_colors = {'kanga': 'steelblue', 'dodson': 'tomato'}
legend_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                          markersize=8, label=a)
                  for a, c in animal_colors.items()]

In [ ]:
# do the plotting
# tgt_measure = 'r_radiusofgyration'
tgt_measure = 'r_meancurvature'
split_mode  = 'median'   # 'fixed' or 'median'

valid_conditions = {
    'dodson': ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': [ 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': ['MC', 'MC_withKanga', 'MC_withKoala'],
    # 'dodson': ['MC_withKanga', ],
    
    'kanga':  ['MC', 'MC_withDodson','MC_withGinger', 'MC_withKoala', 'MC_withVermelho'],
    # 'kanga':  ['MC', 'MC_withDodson', 'MC_withVermelho'],
    # 'kanga':  ['MC_withDodson',],
    # 'kanga':  ['MC', 'MC_withDodson', ],
}

# filter each animal by their valid task conditions
df_filtered = pd.concat([
    df_corr_results[(df_corr_results['recorded_animal']==animal) &
                    (df_corr_results['task_condition'].isin(conds)) & 
                    (df_corr_results['succ_rate']>0.2)]
    for animal, conds in valid_conditions.items()
])

# remove some session that Jing's follower/leader and my follower percentage has big discrepency
if 0:
    fp_median = 0.50
    agreement_threshold = 0.05

    df_filtered_clean = df_filtered[
        ((df_filtered['role'] == 'follower') & (df_filtered['follower_percentage'] > fp_median - agreement_threshold)) |
        ((df_filtered['role'] == 'leader')   & (df_filtered['follower_percentage'] < fp_median + agreement_threshold))
    ]
    
if 0:
    df_follower = df_filtered[df_filtered['role'] == 'follower'].copy()
    df_leader   = df_filtered[df_filtered['role'] == 'leader'].copy()

    def remove_fp_outliers_zscore(df_role, threshold=1.5):
        fp = df_role['follower_percentage'].dropna()
        z_scores = np.abs(stats.zscore(fp))
        outlier_idx = fp.index[z_scores > threshold]
        clean = df_role[~df_role.index.isin(outlier_idx)]
        removed = df_role[df_role.index.isin(outlier_idx)]
        return clean, removed

    z_threshold = 1.5

    df_follower_clean, df_follower_removed = remove_fp_outliers_zscore(df_follower, z_threshold)
    df_leader_clean,   df_leader_removed   = remove_fp_outliers_zscore(df_leader,   z_threshold)

    df_filtered_clean = pd.concat([df_follower_clean, df_leader_clean])

# df_filtered = df_filtered_clean

# ── Figure 1: pooled ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(27, 7))

# Panel 1: by animal
data_kanga  = df_filtered[df_filtered['recorded_animal']=='kanga'][tgt_measure].dropna()
data_dodson = df_filtered[df_filtered['recorded_animal']=='dodson'][tgt_measure].dropna()
plot_violin_with_points(axes[0], [data_kanga, data_dodson], ['kanga','dodson'], 'By animal', tgt_measure,
                        point_colors=[['steelblue']*len(data_kanga), ['tomato']*len(data_dodson)])
axes[0].set_xlabel(stats_label(data_kanga.values, data_dodson.values))

# Panel 2: follower_percentage
df_p2 = df_filtered[[tgt_measure, 'follower_percentage', 'recorded_animal']].dropna()
thr2  = get_threshold(df_p2['follower_percentage'], fixed_val=0.5, mode=split_mode)
data_low_fp  = df_p2[df_p2['follower_percentage'] <= thr2][tgt_measure]
data_high_fp = df_p2[df_p2['follower_percentage'] >  thr2][tgt_measure]
plot_violin_with_points(
    axes[1], [data_low_fp, data_high_fp],
    [f'follower% ≤ {thr2:.2f}', f'follower% > {thr2:.2f}'],
    'By follower percentage', tgt_measure,
    point_colors=get_point_colors(df_p2, tgt_measure, 'follower_percentage', thr2)
)
axes[1].set_xlabel(stats_label(
    data_low_fp.values, data_high_fp.values,
    df_subset=df_p2, measure_col=tgt_measure, split_col='follower_percentage', threshold=thr2
))
axes[1].legend(handles=legend_handles, fontsize=8)

# Panel 3: selfvspartner_pullnum
df_p3 = df_filtered[[tgt_measure, 'selfvspartner_pullnum', 'recorded_animal']].dropna()
thr3  = get_threshold(df_p3['selfvspartner_pullnum'], fixed_val=1.0, mode=split_mode)
data_low_pn  = df_p3[df_p3['selfvspartner_pullnum'] <= thr3][tgt_measure]
data_high_pn = df_p3[df_p3['selfvspartner_pullnum'] >  thr3][tgt_measure]
plot_violin_with_points(
    axes[2], [data_low_pn, data_high_pn],
    [f'pullnum ≤ {thr3:.2f}', f'pullnum > {thr3:.2f}'],
    'By self vs partner pull num', tgt_measure,
    point_colors=get_point_colors(df_p3, tgt_measure, 'selfvspartner_pullnum', thr3)
)
axes[2].set_xlabel(stats_label(
    data_low_pn.values, data_high_pn.values,
    df_subset=df_p3, measure_col=tgt_measure, split_col='selfvspartner_pullnum', threshold=thr3
))
axes[2].legend(handles=legend_handles, fontsize=8)

# Panel 4: selfvspartner_followerPer
df_p4 = df_filtered[[tgt_measure, 'selfvspartner_followerPer', 'recorded_animal']].dropna()
thr4  = get_threshold(df_p4['selfvspartner_followerPer'], fixed_val=1.0, mode=split_mode)
data_low_fpr  = df_p4[df_p4['selfvspartner_followerPer'] <= thr4][tgt_measure]
data_high_fpr = df_p4[df_p4['selfvspartner_followerPer'] >  thr4][tgt_measure]
plot_violin_with_points(
    axes[3], [data_low_fpr, data_high_fpr],
    [f'followerRatio ≤ {thr4:.2f}', f'followerRatio > {thr4:.2f}'],
    'By follower % ratio (self/partner)', tgt_measure,
    point_colors=get_point_colors(df_p4, tgt_measure, 'selfvspartner_followerPer', thr4)
)
axes[3].set_xlabel(stats_label(
    data_low_fpr.values, data_high_fpr.values,
    df_subset=df_p4, measure_col=tgt_measure, split_col='selfvspartner_followerPer', threshold=thr4
))
axes[3].legend(handles=legend_handles, fontsize=8)

# Panel 5: by role
df_p5 = df_filtered[[tgt_measure, 'role', 'recorded_animal']].dropna()
data_follower = df_p5[df_p5['role'] == 'follower'][tgt_measure]
data_leader   = df_p5[df_p5['role'] == 'leader'][tgt_measure]
animal_color_map = {'kanga': 'steelblue', 'dodson': 'tomato'}
colors_follower = df_p5[df_p5['role'] == 'follower']['recorded_animal'].map(animal_color_map).tolist()
colors_leader   = df_p5[df_p5['role'] == 'leader']['recorded_animal'].map(animal_color_map).tolist()
plot_violin_with_points(
    axes[4], [data_follower, data_leader],
    ['follower', 'leader'],
    'By role', tgt_measure,
    point_colors=[colors_follower, colors_leader]
)
axes[4].set_xlabel(stats_label(
    data_follower.values, data_leader.values,
    df_subset=df_p5, measure_col=tgt_measure, split_col='role', threshold='follower'
))
axes[4].legend(handles=legend_handles, fontsize=8)

plt.suptitle(f"{tgt_measure}  [split: {split_mode}]")
plt.tight_layout()
plt.show()

# ── Figure 2: per animal ──────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(2, 4, figsize=(24, 12))

animals    = ['kanga', 'dodson']
row_colors = {'kanga': 'steelblue', 'dodson': 'tomato'}

for irow, animal in enumerate(animals):
    df_animal = df_filtered[df_filtered['recorded_animal'] == animal]
    c = row_colors[animal]

    # col 0: follower_percentage
    df_a2 = df_animal[[tgt_measure, 'follower_percentage']].dropna()
    thr2  = get_threshold(df_a2['follower_percentage'], fixed_val=0.5, mode=split_mode)
    data_low  = df_a2[df_a2['follower_percentage'] <= thr2][tgt_measure]
    data_high = df_a2[df_a2['follower_percentage'] >  thr2][tgt_measure]
    plot_violin_with_points(
        axes2[irow, 0], [data_low, data_high],
        [f'follower% ≤ {thr2:.2f}', f'follower% > {thr2:.2f}'],
        f'{animal} | By follower percentage', tgt_measure,
        point_colors=[[c]*len(data_low), [c]*len(data_high)]
    )
    axes2[irow, 0].set_xlabel(stats_label(data_low.values, data_high.values))

    # col 1: selfvspartner_pullnum
    df_a3 = df_animal[[tgt_measure, 'selfvspartner_pullnum']].dropna()
    thr3  = get_threshold(df_a3['selfvspartner_pullnum'], fixed_val=1.0, mode=split_mode)
    data_low  = df_a3[df_a3['selfvspartner_pullnum'] <= thr3][tgt_measure]
    data_high = df_a3[df_a3['selfvspartner_pullnum'] >  thr3][tgt_measure]
    plot_violin_with_points(
        axes2[irow, 1], [data_low, data_high],
        [f'pullnum ≤ {thr3:.2f}', f'pullnum > {thr3:.2f}'],
        f'{animal} | By self vs partner pull num', tgt_measure,
        point_colors=[[c]*len(data_low), [c]*len(data_high)]
    )
    axes2[irow, 1].set_xlabel(stats_label(data_low.values, data_high.values))

    # col 2: selfvspartner_followerPer
    df_a4 = df_animal[[tgt_measure, 'selfvspartner_followerPer']].dropna()
    thr4  = get_threshold(df_a4['selfvspartner_followerPer'], fixed_val=1.0, mode=split_mode)
    data_low  = df_a4[df_a4['selfvspartner_followerPer'] <= thr4][tgt_measure]
    data_high = df_a4[df_a4['selfvspartner_followerPer'] >  thr4][tgt_measure]
    plot_violin_with_points(
        axes2[irow, 2], [data_low, data_high],
        [f'followerRatio ≤ {thr4:.2f}', f'followerRatio > {thr4:.2f}'],
        f'{animal} | By follower % ratio (self/partner)', tgt_measure,
        point_colors=[[c]*len(data_low), [c]*len(data_high)]
    )
    axes2[irow, 2].set_xlabel(stats_label(data_low.values, data_high.values))

    # col 3: by role
    df_a5 = df_animal[[tgt_measure, 'role', 'recorded_animal']].dropna()
    data_follower_a = df_a5[df_a5['role'] == 'follower'][tgt_measure]
    data_leader_a   = df_a5[df_a5['role'] == 'leader'][tgt_measure]
    plot_violin_with_points(
        axes2[irow, 3], [data_follower_a, data_leader_a],
        ['follower', 'leader'],
        f'{animal} | By role', tgt_measure,
        point_colors=[[c]*len(data_follower_a), [c]*len(data_leader_a)]
    )
    axes2[irow, 3].set_xlabel(stats_label(
        data_follower_a.values, data_leader_a.values,
        df_subset=df_a5, measure_col=tgt_measure, split_col='role', threshold='follower'
    ))

plt.suptitle(f"{tgt_measure} — per animal  [split: {split_mode}]")
plt.tight_layout()

savefig = 1
if savefig:
    figsavefolder = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_Pullfocused_continuousBhv_rewardMotivation_trajectory_forJing/"+\
                    cameraID+"/"

    if not os.path.exists(figsavefolder):
        os.makedirs(figsavefolder)

    fig.savefig(figsavefolder+tgt_measure+'_pooled'+'_split_'+split_mode+'.pdf')
    fig2.savefig(figsavefolder+tgt_measure+'_perAnimal'+'_split_'+split_mode+'.pdf')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

animal_color_map = {'kanga': 'steelblue', 'dodson': 'tomato'}

for role, marker in [('follower', 'o'), ('leader', '^')]:
    df_r = df_filtered[df_filtered['role'] == role].dropna(subset=['follower_percentage'])
    for animal in ['kanga', 'dodson']:
        df_ra = df_r[df_r['recorded_animal'] == animal]
        ax.scatter(
            [role] * len(df_ra),
            df_ra['follower_percentage'],
            color=animal_color_map[animal],
            marker=marker,
            alpha=0.7, s=60, zorder=3,
            label=f'{animal}' if role == 'follower' else ''
        )

ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50%')
ax.axhline(0.55, color='black', linestyle='--', linewidth=1, label='median 55%')

ax.set_ylabel('Follower Percentage')
ax.set_xlabel('Role')
ax.set_title('Follower percentage by role')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()



In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
animal_color_map = {'kanga': 'steelblue', 'dodson': 'tomato'}
for role, marker in [('follower', 'o'), ('leader', '^')]:
    df_r = df_filtered[df_filtered['role'] == role].dropna(subset=['selfvspartner_followerPer'])
    for animal in ['kanga', 'dodson']:
        df_ra = df_r[df_r['recorded_animal'] == animal]
        ax.scatter(
            [role] * len(df_ra),
            df_ra['selfvspartner_followerPer'],
            color=animal_color_map[animal],
            marker=marker,
            alpha=0.7, s=60, zorder=3,
            label=f'{animal}' if role == 'follower' else ''
        )
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='ratio=1')
median_ratio = df_filtered['selfvspartner_followerPer'].median()
ax.axhline(median_ratio, color='black', linestyle='--', linewidth=1, label=f'median={median_ratio:.2f}')
ax.set_ylabel('Self vs Partner Follower Ratio')
ax.set_xlabel('Role')
ax.set_title('Self vs partner follower ratio by role')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# df_filtered

In [ ]:
if 0: 
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import KFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import balanced_accuracy_score
    from scipy import stats

    feature_cols = [
        # 'path_length', 'net_displacement', 'tortuosity', 'mean_speed',
        'mean_curvature', 'radius_of_gyration', 'convex_hull_volume',
        # 'angle_span_net', 'angle_span_total'
    ]

    df_pred_results = []

    for (date, animal), group in df_pca_features.groupby(['date', 'recorded_animal']):

        # ── IPI outlier removal ───────────────────────────────────────────────
        ipi = group['ipi'].values
        finite_ipi = ipi[np.isfinite(ipi)]
        if len(finite_ipi) < 15:
            continue
        Q1, Q3 = np.percentile(finite_ipi, [25, 75])
        IQR = Q3 - Q1
        ipi_mask = (ipi >= Q1 - 1.5*IQR) & (ipi <= Q3 + 1.5*IQR)
        group = group[ipi_mask]

        X = group[feature_cols].values
        y = group['ipi'].values

        # ── drop rows with NaN in features or target ──────────────────────────
        valid = np.isfinite(X).all(axis=1) & np.isfinite(y)
        X, y = X[valid], y[valid]

        if len(X) < 15:
            continue

        # ── median split IPI into binary label ───────────────────────────────
        ipi_median = np.median(y)
        y_binary = (y > ipi_median).astype(int)  # 0=short, 1=long

        # ── 5-fold cross-validated logistic regression ────────────────────────
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        y_true_all, y_pred_all = [], []

        for train_idx, test_idx in kf.split(X):
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X[train_idx])
            X_test  = scaler.transform(X[test_idx])

            model = LogisticRegression(C=1.0, max_iter=1000)
            model.fit(X_train, y_binary[train_idx])
            y_pred_all.extend(model.predict(X_test))
            y_true_all.extend(y_binary[test_idx])

        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)

        acc = balanced_accuracy_score(y_true_all, y_pred_all)

        # ── permutation test: is acc above chance? ────────────────────────────
        n_perms = 100
        perm_accs = []
        for _ in range(n_perms):
            y_shuffled = y_binary.copy()
            np.random.shuffle(y_shuffled)
            y_perm_true, y_perm_pred = [], []
            for train_idx, test_idx in kf.split(X):
                scaler = StandardScaler()
                X_train = scaler.fit_transform(X[train_idx])
                X_test  = scaler.transform(X[test_idx])
                model = LogisticRegression(C=1.0, max_iter=1000)
                model.fit(X_train, y_shuffled[train_idx])
                y_perm_pred.extend(model.predict(X_test))
                y_perm_true.extend(y_shuffled[test_idx])
            perm_accs.append(balanced_accuracy_score(y_perm_true, y_perm_pred))

        perm_p = np.mean(np.array(perm_accs) >= acc)

        df_pred_results.append({
            'date':                date,
            'recorded_animal':     animal,
            'task_condition':      group['task_condition'].iloc[0],
            'role':                group['role'].iloc[0],
            'follower_percentage': group['followerPercentage'].iloc[0],
            'n_trials':            len(y),
            'balanced_acc':        acc,
            'perm_p':              perm_p,
        })

    df_pred_results = pd.DataFrame(df_pred_results)

    # ── summary stats ─────────────────────────────────────────────────────────────
    print("=== Overall ===")
    print(f"  mean balanced acc: {df_pred_results['balanced_acc'].mean():.3f}")
    print(f"  sessions above chance (perm_p<0.05): "
          f"{(df_pred_results['perm_p'] < 0.05).sum()} / {len(df_pred_results)}")

    for animal in ['kanga', 'dodson']:
        df_a = df_pred_results[df_pred_results['recorded_animal'] == animal]
        print(f"\n=== {animal} ===")
        print(f"  mean balanced acc: {df_a['balanced_acc'].mean():.3f}")
        print(f"  sessions above chance: {(df_a['perm_p'] < 0.05).sum()} / {len(df_a)}")

    print("\n=== By role ===")
    for role in ['follower', 'leader']:
        df_r = df_pred_results[df_pred_results['role'] == role]
        print(f"  {role}: mean acc={df_r['balanced_acc'].mean():.3f}, "
              f"n={len(df_r)}, above chance={( df_r['perm_p'] < 0.05).sum()}")

    leaders   = df_pred_results[df_pred_results['role'] == 'leader']['balanced_acc']
    followers = df_pred_results[df_pred_results['role'] == 'follower']['balanced_acc']
    if len(leaders) > 0 and len(followers) > 0:
        stat, p = stats.mannwhitneyu(leaders, followers, alternative='two-sided')
        print(f"\n  Leader vs Follower MWU: p={p:.4f}")

    # ── plotting ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel 1: balanced acc per session, colored by animal
    animal_color_map = {'kanga': 'steelblue', 'dodson': 'tomato'}
    for animal in ['kanga', 'dodson']:
        df_a = df_pred_results[df_pred_results['recorded_animal'] == animal]
        axes[0].scatter(
            [animal] * len(df_a), df_a['balanced_acc'],
            color=animal_color_map[animal], alpha=0.7, s=60, zorder=3
        )
        axes[0].plot(
            [animal, animal],
            [df_a['balanced_acc'].mean(), df_a['balanced_acc'].mean()],
            color=animal_color_map[animal], linewidth=4, alpha=0.9
        )
    axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1, label='chance')
    axes[0].set_ylabel('Balanced Accuracy')
    axes[0].set_title('By animal')
    axes[0].legend(fontsize=8)

    # Panel 2: balanced acc by role, colored by animal
    role_order = ['follower', 'leader']
    for role in role_order:
        df_r = df_pred_results[df_pred_results['role'] == role]
        colors = df_r['recorded_animal'].map(animal_color_map).tolist()
        axes[1].scatter(
            [role] * len(df_r), df_r['balanced_acc'],
            color=colors, alpha=0.7, s=60, zorder=3
        )
        axes[1].plot(
            [role, role],
            [df_r['balanced_acc'].mean(), df_r['balanced_acc'].mean()],
            color='black', linewidth=4, alpha=0.6
        )
    axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=1, label='chance')
    axes[1].set_title('By role')
    axes[1].legend(fontsize=8)

    # Panel 3: distribution of permutation p-values
    axes[2].hist(df_pred_results['perm_p'], bins=15, color='slategray', edgecolor='white')
    axes[2].axvline(0.05, color='red', linestyle='--', linewidth=1.5, label='p=0.05')
    axes[2].set_xlabel('Permutation p-value')
    axes[2].set_ylabel('Session count')
    axes[2].set_title('Sessions above chance')
    axes[2].legend(fontsize=8)

    plt.suptitle('IPI classification from pre-pull PCA trajectory features')
    plt.tight_layout()
    plt.show()

In [ ]:
# give the the top corr value session's info

# ── top N sessions ────────────────────────────────────────────────────────────
top_n = 3

# top_sessions = df_corr_results.nlargest(top_n, tgt_measure)[
top_sessions = df_corr_results.nsmallest(top_n, tgt_measure)[
    ['date', 'recorded_animal', 'task_condition', 
     'follower_percentage', 'selfvspartner_pullnum', 'selfvspartner_followerPer',
     tgt_measure]
].reset_index(drop=True)

# ── add neuron count for each session ────────────────────────────────────────
neuron_counts = []

for _, row in top_sessions.iterrows():
    animal = row['recorded_animal']
    date   = row['date']

    if animal == 'kanga':
        a1, a2 = 'dannon', 'kanga'
    else:
        a1, a2 = 'dodson', 'ginger'

    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix_tgt+\
        '/'+cameraID+'/'+a1+a2+'/'

    try:
        with open(data_saved_subfolder+f'/bhvevents_aligned_FR_allevents_all_dates_{a1}{a2}.pkl', 'rb') as f:
            bhvevents_tmp = pickle.load(f)
        nneurons = len(list(bhvevents_tmp[date][animal+' pull'].keys()))
    except Exception as e:
        print(f"Could not load {date} {animal}: {e}")
        nneurons = np.nan

    neuron_counts.append(nneurons)

top_sessions.insert(3, 'n_neurons', neuron_counts)

print(f"Top {top_n} sessions for {tgt_measure}:")
print(top_sessions.to_string(index=True))

In [ ]:
# ── pick a session to visualize ───────────────────────────────────────────────
# example_animal = 'kanga'
example_animal = 'dodson'
n_traj_to_plot = 5
sigma_smooth   = 10

# ── reload the data for this session ─────────────────────────────────────────
if example_animal == 'kanga':
    a1, a2 = 'dannon', 'kanga'
else:
    a1, a2 = 'dodson', 'ginger'

data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix_tgt+\
    '/'+cameraID+'/'+a1+a2+'/'

with open(data_saved_subfolder+f'/bhvevents_aligned_FR_allevents_all_dates_{a1}{a2}.pkl', 'rb') as f:
    bhvevents_ex = pickle.load(f)
    
with open(data_saved_subfolder+f'/pull_rts_all_dates_{a1}{a2}.pkl', 'rb') as f:
    pull_rts_ex = pickle.load(f)

# ── use the actual keys from bhvevents_ex ─────────────────────────────────────
available_dates = list(bhvevents_ex.keys())
print(f"Available dates ({len(available_dates)}): {available_dates}")

# example_date = available_dates[4]  # change index or set manually e.g. available_dates[3]
example_date = '20250415'
print(f"Plotting: {example_animal} | {example_date}")

# ── extract FR and RT ─────────────────────────────────────────────────────────
FR_pullaligned = bhvevents_ex[example_date][example_animal+' pull']
neuron_ids     = list(FR_pullaligned.keys())
nneurons       = len(neuron_ids)

FR_stack    = np.array([FR_pullaligned[nid]['FR_allevents'] for nid in neuron_ids])
npulls      = FR_stack.shape[2]
nTimePoints = FR_stack.shape[1]

rt_allpulls = np.array(pull_rts_ex[example_date][example_animal])

# ── PCA ───────────────────────────────────────────────────────────────────────
FR_concat = FR_stack.transpose(1, 2, 0).reshape(npulls * nTimePoints, nneurons)
FR_concat = np.nan_to_num(FR_concat, nan=0.0)
pca       = PCA(n_components=3)
FR_pca    = pca.fit_transform(FR_concat)

FR_pca_perpull = [FR_pca[ipull * nTimePoints:(ipull + 1) * nTimePoints, :]
                  for ipull in range(npulls)]

# ── smooth FIRST on full window, then crop to -4 to 0s ───────────────────────
half_tp = int(nTimePoints / 2)
t       = np.linspace(-4, 0, half_tp)

trajs_smoothed = []
for ipull in range(npulls):
    traj_smooth = smooth_trajectory(FR_pca_perpull[ipull], sigma=sigma_smooth)
    trajs_smoothed.append(traj_smooth[:half_tp, :])

# ── get valid trials ──────────────────────────────────────────────────────────
valid_mask = np.isfinite(rt_allpulls)
valid_idx  = np.where(valid_mask)[0]
valid_rt   = rt_allpulls[valid_idx]

low_thresh  = np.percentile(valid_rt, 20)
high_thresh = np.percentile(valid_rt, 80)

idx_short_rt = valid_idx[valid_rt <= low_thresh]
idx_long_rt  = valid_idx[valid_rt >= high_thresh]

# ── find best matching pair on start + end distance ──────────────────────────
# subsample candidates to keep it fast
rng = np.random.default_rng()
max_candidates = 20
candidates_short = rng.choice(idx_short_rt, size=min(max_candidates, len(idx_short_rt)), replace=False)
candidates_long  = rng.choice(idx_long_rt,  size=min(max_candidates, len(idx_long_rt)),  replace=False)

best_dist  = np.inf
pick_short = candidates_short[0]
pick_long  = candidates_long[0]

for i in candidates_short:
    for j in candidates_long:
        traj_i    = trajs_smoothed[i]
        traj_j    = trajs_smoothed[j]
        dist      = (np.linalg.norm(traj_i[0,  :] - traj_j[0,  :]) +
                     np.linalg.norm(traj_i[-1, :] - traj_j[-1, :]))
        if dist < best_dist:
            best_dist  = dist
            pick_short = i
            pick_long  = j

print(f"Short RT trial : pull #{pick_short}, RT={rt_allpulls[pick_short]:.3f}s")
print(f"Long  RT trial : pull #{pick_long},  RT={rt_allpulls[pick_long]:.3f}s")
print(f"Start+end dist : {best_dist:.3f}")

# ── compute 3D convex hull volume ─────────────────────────────────────────────
from scipy.spatial import ConvexHull
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

def convex_hull_volume_3d(traj):
    try:
        return ConvexHull(traj).volume
    except Exception:
        return np.nan

def draw_hull_2d(ax, points, color, alpha_fill=0.15):
    try:
        hull  = ConvexHull(points)
        verts = np.append(hull.vertices, hull.vertices[0])
        ax.fill(points[hull.vertices, 0], points[hull.vertices, 1],
                color=color, alpha=alpha_fill)
        ax.plot(points[verts, 0], points[verts, 1],
                color=color, linewidth=1.5, alpha=0.8)
        return hull.volume
    except Exception:
        return np.nan

traj_short   = trajs_smoothed[pick_short]
traj_long    = trajs_smoothed[pick_long]
vol_short_3d = convex_hull_volume_3d(traj_short)
vol_long_3d  = convex_hull_volume_3d(traj_long)

print(f"Short RT 3D hull volume: {vol_short_3d:.3f}")
print(f"Long  RT 3D hull volume: {vol_long_3d:.3f}")

# ── plot ──────────────────────────────────────────────────────────────────────
short_color = 'royalblue'
long_color  = 'tomato'
pc_pairs    = [(0, 1), (0, 2), (1, 2)]
pc_labels   = ['PC1', 'PC2', 'PC3']

fig = plt.figure(figsize=(18, 5))

# ── 3D panel ──────────────────────────────────────────────────────────────────
ax3d = fig.add_subplot(1, 4, 1, projection='3d')

for traj, color, label in [
    (traj_short, short_color, f'short RT={rt_allpulls[pick_short]:.2f}s'),
    (traj_long,  long_color,  f'long  RT={rt_allpulls[pick_long]:.2f}s'),
]:
    ax3d.plot(traj[:, 0], traj[:, 1], traj[:, 2],
              color=color, linewidth=2.0, label=label)
    ax3d.scatter(*traj[0],  color=color, s=50, marker='o', zorder=5)
    ax3d.scatter(*traj[-1], color=color, s=70, marker='*', zorder=5)

ax3d.set_xlabel('PC1'); ax3d.set_ylabel('PC2'); ax3d.set_zlabel('PC3')
ax3d.set_title(f'3D trajectory\n'
               f'3D hull: short={vol_short_3d:.2f}, long={vol_long_3d:.2f}')
ax3d.legend(fontsize=7)

# ── compute global axis limits across all 2D panels ───────────────────────────
all_traj_points = np.vstack([traj_short, traj_long])  # (2*half_tp, 3)
global_min = all_traj_points.min() - 0.3 * np.abs(all_traj_points).max()
global_max = all_traj_points.max() + 0.3 * np.abs(all_traj_points).max()
# make it symmetric so equal aspect looks nice
# abs_lim = max(abs(global_min), abs(global_max))
# lim     = [-abs_lim, abs_lim]

# ── 2D projection panels ──────────────────────────────────────────────────────
for ipanel, (pcx, pcy) in enumerate(pc_pairs):
    ax = fig.add_subplot(1, 4, ipanel + 2)

    for traj, color in [(traj_short, short_color), (traj_long, long_color)]:
        points = traj[:, [pcx, pcy]]
        draw_hull_2d(ax, points, color=color, alpha_fill=0.15)
        ax.plot(traj[:, pcx], traj[:, pcy], color=color, linewidth=2.0, zorder=4)
        ax.scatter(traj[0,  pcx], traj[0,  pcy], color=color, s=50, marker='o', zorder=5)
        ax.scatter(traj[-1, pcx], traj[-1, pcy], color=color, s=70, marker='*', zorder=5)

    if ipanel == 0:
        ax.annotate('-4s', xy=(traj_short[0, pcx], traj_short[0, pcy]),
                    xytext=(5, 5), textcoords='offset points', fontsize=7, color='gray')
        ax.annotate(' 0s', xy=(traj_short[-1, pcx], traj_short[-1, pcy]),
                    xytext=(5, 5), textcoords='offset points', fontsize=7, color='gray')

    # consistent square axes across all panels
    # ax.set_xlim(lim)
    # ax.set_ylim(lim)
    # ax.set_aspect('equal')

    ax.set_xlabel(pc_labels[pcx])
    ax.set_ylabel(pc_labels[pcy])
    ax.set_title(f'{pc_labels[pcx]} vs {pc_labels[pcy]}')
    ax.axhline(0, color='lightgray', linewidth=0.5, zorder=0)
    ax.axvline(0, color='lightgray', linewidth=0.5, zorder=0)

    if ipanel == 0:
        legend_elements = [
            Line2D([0], [0], color=short_color, linewidth=2.0,
                   label=f'short RT={rt_allpulls[pick_short]:.3f}s | 3D vol={vol_short_3d:.2f}'),
            Line2D([0], [0], color=long_color,  linewidth=2.0,
                   label=f'long  RT={rt_allpulls[pick_long]:.3f}s | 3D vol={vol_long_3d:.2f}'),
            Line2D([0], [0], marker='o', color='gray', linewidth=0,
                   markersize=6, label='start (-4s)'),
            Line2D([0], [0], marker='*', color='gray', linewidth=0,
                   markersize=8, label='end (0s)'),
        ]
        ax.legend(handles=legend_elements, fontsize=7, loc='best')

plt.suptitle(f'Example trials | {example_animal} | {example_date} | '
             f'sigma={sigma_smooth} | {nneurons} neurons\n'
             f'Note: 2D panels are projections; hull volume computed in full 3D',
             y=1.02)
plt.tight_layout()


savefig = 1
if savefig:
    figsavefolder = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_Pullfocused_continuousBhv_rewardMotivation_trajectory_forJing/"+\
                    cameraID+"/"

    if not os.path.exists(figsavefolder):
        os.makedirs(figsavefolder)

    fig.savefig(figsavefolder+'example_single_trial_trace_'+example_animal+'_'+example_date+'.pdf')


In [ ]:
# ── pick a session to visualize ───────────────────────────────────────────────
example_animal = 'dodson'
sigma_smooth   = 10

# ── reload the data for this session ─────────────────────────────────────────
if example_animal == 'kanga':
    a1, a2 = 'dannon', 'kanga'
else:
    a1, a2 = 'dodson', 'ginger'

data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix_tgt+\
    '/'+cameraID+'/'+a1+a2+'/'

with open(data_saved_subfolder+f'/bhvevents_aligned_FR_allevents_all_dates_{a1}{a2}.pkl', 'rb') as f:
    bhvevents_ex = pickle.load(f)

with open(data_saved_subfolder+f'/pull_rts_all_dates_{a1}{a2}.pkl', 'rb') as f:
    pull_rts_ex = pickle.load(f)

available_dates = list(bhvevents_ex.keys())
print(f"Available dates ({len(available_dates)}): {available_dates}")

example_date = '20250227'
print(f"Plotting: {example_animal} | {example_date}")

# ── extract FR and RT ─────────────────────────────────────────────────────────
FR_pullaligned = bhvevents_ex[example_date][example_animal+' pull']
neuron_ids     = list(FR_pullaligned.keys())
nneurons       = len(neuron_ids)

FR_stack    = np.array([FR_pullaligned[nid]['FR_allevents'] for nid in neuron_ids])
npulls      = FR_stack.shape[2]
nTimePoints = FR_stack.shape[1]

rt_allpulls = np.array(pull_rts_ex[example_date][example_animal])

# ── PCA ───────────────────────────────────────────────────────────────────────
FR_concat = FR_stack.transpose(1, 2, 0).reshape(npulls * nTimePoints, nneurons)
FR_concat = np.nan_to_num(FR_concat, nan=0.0)
pca       = PCA(n_components=3)
FR_pca    = pca.fit_transform(FR_concat)

FR_pca_perpull = [FR_pca[ipull * nTimePoints:(ipull + 1) * nTimePoints, :]
                  for ipull in range(npulls)]

# ── smooth FIRST on full window, then crop to -4 to 0s ───────────────────────
half_tp = int(nTimePoints / 2)

trajs_smoothed = []
for ipull in range(npulls):
    traj_smooth = smooth_trajectory(FR_pca_perpull[ipull], sigma=sigma_smooth)
    trajs_smoothed.append(traj_smooth[:half_tp, :])

# ── get valid trials ──────────────────────────────────────────────────────────
valid_mask = np.isfinite(rt_allpulls)
valid_idx  = np.where(valid_mask)[0]
valid_rt   = rt_allpulls[valid_idx]

low_thresh  = np.percentile(valid_rt, 20)
high_thresh = np.percentile(valid_rt, 80)

idx_short_rt = valid_idx[valid_rt <= low_thresh]
idx_long_rt  = valid_idx[valid_rt >= high_thresh]

# ── helper functions ──────────────────────────────────────────────────────────
def local_curvature(traj):
    """Per-timepoint curvature (turning angle). Returns array of length nTimePoints-2."""
    steps = np.diff(traj, axis=0)
    norms = np.linalg.norm(steps, axis=1)
    valid = norms > 0
    norms_safe   = np.where(norms > 0, norms, 1)
    steps_normed = steps / norms_safe[:, np.newaxis]
    dots  = np.sum(steps_normed[:-1] * steps_normed[1:], axis=1)
    dots  = np.clip(dots, -1, 1)
    angles = np.arccos(dots).copy()
    angles[~(valid[:-1] & valid[1:])] = np.nan
    return angles

def plot_traj_single_color(ax, traj, pcx, pcy, color):
    ax.plot(traj[:, pcx], traj[:, pcy],
            color=color, linewidth=2.5, zorder=4)

# ── find pair: short RT has HIGHER curvature than long RT ────────────────────
# also maximize curvature difference + minimize start/end spatial distance
rng = np.random.default_rng()
max_candidates   = 30
candidates_short = rng.choice(idx_short_rt, size=min(max_candidates, len(idx_short_rt)), replace=False)
candidates_long  = rng.choice(idx_long_rt,  size=min(max_candidates, len(idx_long_rt)),  replace=False)

scores = []
pairs  = []
for i in candidates_short:
    for j in candidates_long:
        curv_i = np.nanmean(local_curvature(trajs_smoothed[i]))
        curv_j = np.nanmean(local_curvature(trajs_smoothed[j]))
        if np.isnan(curv_i) or np.isnan(curv_j):
            continue

        # ── enforce direction: short RT must have HIGHER curvature ────────
        if curv_i <= curv_j:
            continue

        curv_diff    = curv_i - curv_j  # signed, always positive now
        start_dist   = np.linalg.norm(trajs_smoothed[i][0,  :] - trajs_smoothed[j][0,  :])
        end_dist     = np.linalg.norm(trajs_smoothed[i][-1, :] - trajs_smoothed[j][-1, :])
        spatial_dist = start_dist + end_dist
        scores.append((curv_diff, spatial_dist))
        pairs.append((i, j))

if len(pairs) == 0:
    print("No valid pairs found with correct curvature direction — try increasing max_candidates")
else:
    scores = np.array(scores)
    curv_diffs    = scores[:, 0]
    spatial_dists = scores[:, 1]

    # normalize each to [0, 1]
    curv_norm    = (curv_diffs    - curv_diffs.min())    / (curv_diffs.max()    - curv_diffs.min()    + 1e-10)
    spatial_norm = (spatial_dists - spatial_dists.min()) / (spatial_dists.max() - spatial_dists.min() + 1e-10)

    # maximize curvature diff, minimize spatial dist
    weight_curv    = 0.5
    weight_spatial = 0.5
    combined = weight_curv * curv_norm - weight_spatial * spatial_norm

    best_idx          = np.argmax(combined)
    pick_short, pick_long = pairs[best_idx]

    print(f"Short RT trial : pull #{pick_short}, RT={rt_allpulls[pick_short]:.3f}s")
    print(f"Long  RT trial : pull #{pick_long},  RT={rt_allpulls[pick_long]:.3f}s")
    print(f"Curvature diff : {curv_diffs[best_idx]:.3f}  (short > long ✓)")
    print(f"Spatial dist   : {spatial_dists[best_idx]:.3f}")

    # ── compute metrics ───────────────────────────────────────────────────────
    from matplotlib.lines import Line2D

    traj_short      = trajs_smoothed[pick_short]
    traj_long       = trajs_smoothed[pick_long]
    curv_short      = local_curvature(traj_short)
    curv_long       = local_curvature(traj_long)
    mean_curv_short = np.nanmean(curv_short)
    mean_curv_long  = np.nanmean(curv_long)

    print(f"Short RT | mean curvature={mean_curv_short:.3f}")
    print(f"Long  RT | mean curvature={mean_curv_long:.3f}")

    # ── plot ──────────────────────────────────────────────────────────────────
    short_color = 'royalblue'
    long_color  = 'tomato'
    pc_pairs    = [(0, 1), (0, 2), (1, 2)]
    pc_labels   = ['PC1', 'PC2', 'PC3']

    fig = plt.figure(figsize=(22, 5))

    # ── 3D panel ──────────────────────────────────────────────────────────────
    ax3d = fig.add_subplot(1, 5, 1, projection='3d')
    for traj, color, label in [
        (traj_short, short_color,
         f'short RT={rt_allpulls[pick_short]:.2f}s | curv={mean_curv_short:.3f}'),
        (traj_long,  long_color,
         f'long  RT={rt_allpulls[pick_long]:.2f}s  | curv={mean_curv_long:.3f}'),
    ]:
        ax3d.plot(traj[:, 0], traj[:, 1], traj[:, 2],
                  color=color, linewidth=2.0, label=label)
        ax3d.scatter(*traj[0],  color=color, s=50, marker='o', zorder=5)
        ax3d.scatter(*traj[-1], color=color, s=70, marker='*', zorder=5)
    ax3d.set_xlabel('PC1'); ax3d.set_ylabel('PC2'); ax3d.set_zlabel('PC3')
    ax3d.set_title(f'3D trajectory\n'
                   f'short curv={mean_curv_short:.3f}\n'
                   f'long  curv={mean_curv_long:.3f}')
    ax3d.legend(fontsize=7)

    # ── 2D projection panels ──────────────────────────────────────────────────
    for ipanel, (pcx, pcy) in enumerate(pc_pairs):
        ax = fig.add_subplot(1, 5, ipanel + 2)

        for traj, color in [
            (traj_short, short_color),
            (traj_long,  long_color),
        ]:
            plot_traj_single_color(ax, traj, pcx, pcy, color)
            ax.scatter(traj[0,  pcx], traj[0,  pcy], color=color, s=50, marker='o', zorder=6)
            ax.scatter(traj[-1, pcx], traj[-1, pcy], color=color, s=70, marker='*', zorder=6)

        if ipanel == 0:
            ax.annotate('-4s', xy=(traj_short[0, pcx], traj_short[0, pcy]),
                        xytext=(5, 5), textcoords='offset points', fontsize=7, color='gray')
            ax.annotate(' 0s', xy=(traj_short[-1, pcx], traj_short[-1, pcy]),
                        xytext=(5, 5), textcoords='offset points', fontsize=7, color='gray')
            legend_elements = [
                Line2D([0], [0], color=short_color, linewidth=2.0,
                       label=f'short RT={rt_allpulls[pick_short]:.3f}s | curv={mean_curv_short:.3f}'),
                Line2D([0], [0], color=long_color,  linewidth=2.0,
                       label=f'long  RT={rt_allpulls[pick_long]:.3f}s | curv={mean_curv_long:.3f}'),
                Line2D([0], [0], marker='o', color='gray', linewidth=0,
                       markersize=6, label='start (-4s)'),
                Line2D([0], [0], marker='*', color='gray', linewidth=0,
                       markersize=8, label='end (0s)'),
            ]
            ax.legend(handles=legend_elements, fontsize=7, loc='best')

        ax.set_xlabel(pc_labels[pcx])
        ax.set_ylabel(pc_labels[pcy])
        ax.set_title(f'{pc_labels[pcx]} vs {pc_labels[pcy]}')
        ax.axhline(0, color='lightgray', linewidth=0.5, zorder=0)
        ax.axvline(0, color='lightgray', linewidth=0.5, zorder=0)

    # ── curvature time series panel ───────────────────────────────────────────
    ax_curv = fig.add_subplot(1, 5, 5)
    t_curv  = np.linspace(-4, 0, len(curv_short))
    ax_curv.plot(t_curv, curv_short, color=short_color, linewidth=2.0,
                 label=f'short RT | mean={mean_curv_short:.3f}')
    ax_curv.plot(t_curv, curv_long,  color=long_color,  linewidth=2.0,
                 label=f'long  RT | mean={mean_curv_long:.3f}')
    ax_curv.axhline(mean_curv_short, color=short_color, linewidth=1.0,
                    linestyle='--', alpha=0.6)
    ax_curv.axhline(mean_curv_long,  color=long_color,  linewidth=1.0,
                    linestyle='--', alpha=0.6)
    ax_curv.set_xlabel('Time (s)')
    ax_curv.set_ylabel('Local curvature (rad)')
    ax_curv.set_title('Curvature over time')
    ax_curv.legend(fontsize=7)
    ax_curv.axhline(0, color='lightgray', linewidth=0.5)

    plt.suptitle(f'Example trials | {example_animal} | {example_date} | '
                 f'sigma={sigma_smooth} | {nneurons} neurons',
                 y=1.02)
    plt.tight_layout()

    savefig = 1
    if savefig:
        figsavefolder = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_Pullfocused_continuousBhv_rewardMotivation_trajectory_forJing/"+\
                        cameraID+"/"
        if not os.path.exists(figsavefolder):
            os.makedirs(figsavefolder)
        fig.savefig(figsavefolder+'example_single_trial_trace_'+example_animal+'_'+example_date+'.pdf')

In [ ]:
angles